# Encore: YAMNet sound labels and musical-role suggestions


## Setup

In [1]:
import os, sys
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TensorFlow's start-up messages

ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    # Install dependencies directly (no requirements.txt needed on Colab).
    !pip install -q 'tensorflow>=2.21,<2.22' 'numpy>=1.26' 'scipy>=1.12' imageio-ffmpeg pandas scikit-learn
elif not Path("requirements.txt").is_file():
    raise SystemExit(f"Run this notebook from the repo root. Current folder: {Path.cwd()}")

if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    print(f"WARNING: Python {sys.version.split()[0]}. TensorFlow needs 3.10 to 3.13.")

Path("encore_audio").mkdir(exist_ok=True)   # Part A writes its files into these two folders
Path("scripts").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)
Path("data/clips/esc50").mkdir(parents=True, exist_ok=True)
Path("data/my_recordings").mkdir(parents=True, exist_ok=True)
Path("data/labels").mkdir(parents=True, exist_ok=True)
Path("data/features").mkdir(parents=True, exist_ok=True)
Path("data/feedback").mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from IPython.display import Audio, display

print("Python", sys.version.split()[0], "| folder:", Path.cwd())
if ON_COLAB:
    print("Running on Colab. No GitHub repo needed: the notebook creates everything.")

Python 3.13.18 | folder: /content
Running on Colab. No GitHub repo needed: the notebook creates everything.


# Part A: the code

## A1. The classifier (`encore_audio/`)

This is everything the backend needs. Data flows through the files in this order:

```
audio file -> audio_io -> yamnet_model  -> top 5 sound labels
                       -> descriptors   -> role_prior (now) or role_model (after training) -> top 2 roles
                                           all assembled by classify.py
```

### `encore_audio/config.py`: settings

The five roles and every constant that training, evaluation and inference must agree on, in one place.

In [2]:
%%writefile encore_audio/config.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Shared constants.

Everything that training, evaluation and inference must agree on lives here, so there is
exactly one place to change it. If you change anything in the PREPROCESSING block, bump
PREPROCESS_VERSION and re-run feature extraction and training: a role model trained on one
preprocessing version is ignored at inference when the versions differ.
"""
from __future__ import annotations

import os
from pathlib import Path

VERSION = "0.1.0"

# --------------------------------------------------------------------------- paths
PROJECT_ROOT = Path(__file__).resolve().parent.parent
MODELS_DIR = Path(os.environ.get("ENCORE_MODELS_DIR", PROJECT_ROOT / "models"))
YAMNET_DIR = Path(os.environ.get("ENCORE_YAMNET_DIR", MODELS_DIR / "yamnet"))
ROLE_MODEL_NPZ = MODELS_DIR / "role_model.npz"
ROLE_MODEL_JSON = MODELS_DIR / "role_model.json"

# --------------------------------------------------------------------------- preprocessing
PREPROCESS_VERSION = "pre-1"
SAMPLE_RATE = 16_000            # YAMNet's required sample rate
MAX_DECODE_SECONDS = 60.0       # never decode more than this from a file
MAX_ANALYSIS_SECONDS = 10.0     # after trimming, analyse at most this much
MIN_FILE_SECONDS = 0.05         # shorter than this is rejected as "too_short"
MIN_YAMNET_SAMPLES = 16_000     # short clips are zero-padded to 1.0 s for YAMNet
SILENCE_PEAK_DBFS = -50.0       # peak below this is rejected as "silent"
TRIM_BELOW_PEAK_DB = 40.0       # leading/trailing audio this far below the loudest frame is trimmed
TRIM_PRE_ROLL_S = 0.05
TRIM_POST_ROLL_S = 0.10
TARGET_PEAK = 0.9               # peak normalisation target (linear)

# --------------------------------------------------------------------------- review thresholds
LOW_LABEL_SCORE = 0.20          # top YAMNet score below this -> "low_label_confidence"
WEAK_ROLE_SCORE = 0.30          # a role score below this is "weak"; if even the top role is
                                # weak, the result gets the "weak_role_evidence" flag
STRONG_ROLE_SCORE = 0.60        # a role score at or above this is "strong"
CLIPPED_FRACTION = 0.001        # more than 0.1 % of samples at full scale -> "clipped"
LOW_CONTRAST_DB = 15.0          # loudest vs quietest parts closer than this -> "steady_or_noisy"

# --------------------------------------------------------------------------- the five roles
# (role_id, display name, description). The ORDER here is the canonical order everywhere:
# label files, model weights and the JSON output all use it.
ROLES: list[tuple[str, str, str]] = [
    ("percussion", "Percussion", "Hits, impacts, thumps, claps, and snaps"),
    ("rhythmic", "Rhythmic", "Repeating patterns: ticking, rattling, shaking"),
    ("melodic", "Melodic", "Pitched or tonal sounds, hums, whistles, bass"),
    ("ambient", "Ambient", "Sustained backgrounds, drones, textures, atmosphere"),
    ("transition", "Transition", "Effects, sweeps, bursts, and one-shot accents"),
]
ROLE_IDS: list[str] = [r[0] for r in ROLES]
ROLE_NAMES: dict[str, str] = {r[0]: r[1] for r in ROLES}
ROLE_DESCRIPTIONS: dict[str, str] = {r[0]: r[2] for r in ROLES}

Writing encore_audio/config.py


### `encore_audio/audio_io.py`: loading and cleaning audio

Decodes any format to 16 kHz mono with ffmpeg, rejects silent / corrupt / tiny files, trims silence and normalises the level. Every clip, in training and at inference, goes through this same function.

In [3]:
%%writefile encore_audio/audio_io.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Audio loading and preprocessing.

One code path for every format and for training, evaluation and inference:

    file (m4a / wav / mp3 / anything ffmpeg reads)
      -> ffmpeg -> 16 kHz mono float32
      -> remove DC offset
      -> reject if empty / too short / silent
      -> trim leading and trailing near-silence
      -> cap at MAX_ANALYSIS_SECONDS
      -> peak-normalise
      -> (for YAMNet only) zero-pad to at least 1.0 s

ffmpeg is the only decoder, so a WAV and an M4A of the same sound go through identical
resampling and channel mixing.
"""
from __future__ import annotations

import shutil
import subprocess
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np

from . import config as C


class AudioRejected(Exception):
    """The file cannot be analysed. `code` is machine-readable, `message` is for humans."""

    def __init__(self, code: str, message: str):
        super().__init__(message)
        self.code = code
        self.message = message


@dataclass
class PreparedAudio:
    active: np.ndarray        # trimmed, normalised audio. Acoustic descriptors use this.
    yamnet_input: np.ndarray  # `active`, zero-padded to >= 1.0 s. YAMNet uses this.
    info: dict = field(default_factory=dict)   # JSON-serialisable facts about the recording
    flags: list = field(default_factory=list)  # review flags raised by preprocessing


_FFMPEG: str | None = None


def find_ffmpeg() -> str:
    """Use the ffmpeg binary that ships with the `imageio-ffmpeg` pip package, falling back
    to a system ffmpeg. This keeps the project runnable without a system-wide install."""
    global _FFMPEG
    if _FFMPEG:
        return _FFMPEG
    try:
        import imageio_ffmpeg

        _FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        _FFMPEG = shutil.which("ffmpeg")
    if not _FFMPEG:
        raise RuntimeError(
            "ffmpeg not found. Run `pip install imageio-ffmpeg` (bundles a binary) "
            "or install ffmpeg with your package manager."
        )
    return _FFMPEG


def decode_to_mono_16k(path: str | Path) -> np.ndarray:
    """Decode any audio file to 16 kHz mono float32 in [-1, 1]."""
    path = Path(path)
    if not path.is_file():
        raise AudioRejected("file_not_found", f"No such file: {path}")
    if path.stat().st_size == 0:
        raise AudioRejected("empty", "The file is empty (0 bytes).")
    cmd = [
        find_ffmpeg(), "-nostdin", "-v", "error",
        "-t", str(C.MAX_DECODE_SECONDS), "-i", str(path),
        "-vn", "-ac", "1", "-ar", str(C.SAMPLE_RATE), "-f", "f32le", "-",
    ]
    try:
        proc = subprocess.run(cmd, capture_output=True, timeout=120)
    except subprocess.TimeoutExpired:
        raise AudioRejected("decode_failed", "ffmpeg timed out while decoding the file.")
    if proc.returncode != 0 or len(proc.stdout) < 4:
        detail = proc.stderr.decode("utf-8", "replace").strip().splitlines()
        raise AudioRejected(
            "decode_failed",
            "Could not decode this file as audio" + (f": {detail[-1]}" if detail else "."),
        )
    usable = len(proc.stdout) - (len(proc.stdout) % 4)
    return np.frombuffer(proc.stdout[:usable], dtype="<f4").astype(np.float32)


def frame_rms(x: np.ndarray, win: int, hop: int) -> np.ndarray:
    """RMS level in windows of `win` samples taken every `hop` samples."""
    if len(x) < win:
        x = np.pad(x, (0, win - len(x)))
    csum = np.concatenate(([0.0], np.cumsum(x.astype(np.float64) ** 2)))
    starts = np.arange(0, len(x) - win + 1, hop)
    return np.sqrt(np.maximum(csum[starts + win] - csum[starts], 0.0) / win)


def _db(v) -> float:
    return float(20.0 * np.log10(max(float(v), 1e-12)))


def prepare_waveform(raw: np.ndarray) -> PreparedAudio:
    """Validate, trim and normalise a 16 kHz mono waveform. Raises AudioRejected."""
    sr = C.SAMPLE_RATE
    if raw.size == 0:
        raise AudioRejected("empty", "The file contains no audio samples.")
    if not np.all(np.isfinite(raw)):
        raise AudioRejected("decode_failed", "The decoded audio contains invalid samples.")
    file_seconds = raw.size / sr
    if file_seconds < C.MIN_FILE_SECONDS:
        raise AudioRejected(
            "too_short", f"Only {file_seconds * 1000:.0f} ms of audio. Record at least 50 ms."
        )

    x = raw.astype(np.float64) - float(np.mean(raw))  # remove DC offset
    peak = float(np.max(np.abs(x)))
    if _db(peak) < C.SILENCE_PEAK_DBFS:
        raise AudioRejected(
            "silent",
            f"The recording is silent or far too quiet (peak {_db(peak):.0f} dBFS). "
            "Check the microphone and record again.",
        )

    flags: list[str] = []
    clipped_fraction = float(np.mean(np.abs(raw) >= 0.999))
    if clipped_fraction > C.CLIPPED_FRACTION:
        flags.append("clipped")

    # Frame levels: 20 ms windows every 10 ms.
    win, hop = int(0.020 * sr), int(0.010 * sr)
    env_db = 20.0 * np.log10(frame_rms(x, win, hop) + 1e-9)
    top_db = float(env_db.max())
    floor_db = float(np.percentile(env_db, 10))
    contrast_db = top_db - floor_db
    if file_seconds >= 0.5 and contrast_db < C.LOW_CONTRAST_DB:
        # Honest ambiguity: from one recording we cannot tell a continuous sound (a hum,
        # running water) from a short sound buried in background noise.
        flags.append("steady_or_noisy")

    # Trim leading / trailing audio that is far below the loudest frame.
    active = np.flatnonzero(env_db >= top_db - C.TRIM_BELOW_PEAK_DB)
    start = max(0, int(active[0]) * hop - int(C.TRIM_PRE_ROLL_S * sr))
    end = min(x.size, int(active[-1]) * hop + win + int(C.TRIM_POST_ROLL_S * sr))
    trimmed = x[start:end].copy()

    max_len = int(C.MAX_ANALYSIS_SECONDS * sr)
    if trimmed.size > max_len:
        trimmed = trimmed[:max_len]
        fade = int(0.020 * sr)
        trimmed[-fade:] *= np.linspace(1.0, 0.0, fade)
        flags.append("truncated")

    gain = C.TARGET_PEAK / max(float(np.max(np.abs(trimmed))), 1e-9)
    active_wave = (trimmed * gain).astype(np.float32)

    yamnet_input = active_wave
    if yamnet_input.size < C.MIN_YAMNET_SAMPLES:
        yamnet_input = np.pad(yamnet_input, (0, C.MIN_YAMNET_SAMPLES - yamnet_input.size))

    info = {
        "file_duration_s": round(file_seconds, 3),
        "analysed_duration_s": round(active_wave.size / sr, 3),
        "sample_rate": sr,
        "peak_dbfs": round(_db(peak), 1),
        "noise_floor_dbfs": round(floor_db, 1),
        "contrast_db": round(contrast_db, 1),
        "clipped_fraction": round(clipped_fraction, 5),
        "gain_applied_db": round(_db(gain), 1),
    }
    return PreparedAudio(active=active_wave, yamnet_input=yamnet_input, info=info, flags=flags)


def load_audio(path: str | Path) -> PreparedAudio:
    """decode + prepare. This is the single entry point used everywhere."""
    return prepare_waveform(decode_to_mono_16k(path))

Writing encore_audio/audio_io.py


### `encore_audio/yamnet_model.py`: YAMNet

Downloads Google's pretrained YAMNet once, loads it, and runs it. Returns 521 class scores and a 1,024-number embedding for each 0.96 s frame.

In [4]:
%%writefile encore_audio/yamnet_model.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Loading and running Google's pretrained YAMNet.

The model is the official TF2 SavedModel published as `google/yamnet/1`. It is downloaded
once into `models/yamnet/` and loaded from disk with plain `tf.saved_model.load`, so:

  * there is no dependency on the `tensorflow_hub` package (its current release fails to
    import on new setuptools versions because it still uses `pkg_resources`),
  * nothing is cached in /tmp (which Fedora wipes on reboot),
  * after the first download the project runs fully offline.

YAMNet takes a 16 kHz mono waveform and slides a 0.96 s window over it every 0.48 s.
For each window ("frame") it returns:
  scores      521 values, one per AudioSet class (independent sigmoids, NOT a softmax)
  embeddings  1024 values: the layer just before the classifier
"""
from __future__ import annotations

import csv
import shutil
import tarfile
import tempfile
import threading
import urllib.request
from pathlib import Path

import numpy as np

from . import config as C

YAMNET_HANDLE = "google/yamnet/1"
# Tried in order. All three serve the same official archive.
DOWNLOAD_URLS = [
    "https://tfhub.dev/google/yamnet/1?tf-hub-format=compressed",
    "https://storage.googleapis.com/tfhub-modules/google/yamnet/1.tar.gz",
    "https://www.kaggle.com/api/v1/models/google/yamnet/tensorFlow2/yamnet/1/download",
]

_lock = threading.Lock()
_model = None
_class_names: list[str] | None = None


def yamnet_is_downloaded(dest: Path | None = None) -> bool:
    dest = Path(dest or C.YAMNET_DIR)
    return (dest / "saved_model.pb").is_file() and (dest / "variables").is_dir()


def download_yamnet(dest: Path | None = None, verbose: bool = True) -> Path:
    """Download and unpack the YAMNet SavedModel into `dest` (default: models/yamnet)."""
    dest = Path(dest or C.YAMNET_DIR)
    if yamnet_is_downloaded(dest):
        return dest
    errors = []
    for url in DOWNLOAD_URLS:
        tmpdir = Path(tempfile.mkdtemp(prefix="yamnet_dl_"))
        try:
            if verbose:
                print(f"Downloading YAMNet from {url} ...")
            archive = tmpdir / "yamnet.tar.gz"
            req = urllib.request.Request(url, headers={"User-Agent": "encore-yamnet/1.0"})
            with urllib.request.urlopen(req, timeout=120) as resp, open(archive, "wb") as out:
                shutil.copyfileobj(resp, out)
            unpacked = tmpdir / "unpacked"
            unpacked.mkdir()
            with tarfile.open(archive) as tar:
                tar.extractall(unpacked, filter="data")  # "data" filter blocks path tricks
            # The archive root is the SavedModel itself; tolerate one level of nesting.
            roots = [p.parent for p in unpacked.rglob("saved_model.pb")]
            if not roots:
                raise RuntimeError("archive did not contain saved_model.pb")
            dest.parent.mkdir(parents=True, exist_ok=True)
            if dest.exists():
                shutil.rmtree(dest)
            shutil.move(str(roots[0]), str(dest))
            if verbose:
                print(f"YAMNet saved to {dest}")
            return dest
        except Exception as exc:  # try the next mirror
            errors.append(f"  {url}\n    -> {type(exc).__name__}: {exc}")
        finally:
            shutil.rmtree(tmpdir, ignore_errors=True)
    raise RuntimeError(
        "Could not download YAMNet. Tried:\n" + "\n".join(errors) + "\n\n"
        "Manual fix: download the TensorFlow 2 version of google/yamnet from "
        "https://www.kaggle.com/models/google/yamnet , unpack it, and place the folder that "
        f"contains saved_model.pb at:\n  {dest}"
    )


def load_yamnet():
    """Load the model once per process (thread-safe). Returns (model, class_names)."""
    global _model, _class_names
    if _model is not None:
        return _model, _class_names
    with _lock:
        if _model is not None:
            return _model, _class_names
        path = download_yamnet(verbose=True)
        import tensorflow as tf  # imported lazily: it takes a few seconds

        model = tf.saved_model.load(str(path))
        class_map = model.class_map_path().numpy().decode("utf-8")
        with open(class_map, newline="", encoding="utf-8") as f:
            names = [row["display_name"] for row in csv.DictReader(f)]
        if len(names) != 521:
            raise RuntimeError(f"Expected 521 YAMNet classes, found {len(names)}.")
        _model, _class_names = model, names
    return _model, _class_names


def class_names() -> list[str]:
    return load_yamnet()[1]


def run_yamnet(waveform: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Run YAMNet on a 16 kHz mono float32 waveform in [-1, 1].

    Returns (scores [n_frames, 521], embeddings [n_frames, 1024]).
    """
    model, _ = load_yamnet()
    out = model(np.asarray(waveform, dtype=np.float32))
    if isinstance(out, dict):
        # Models exported with the current version of Google's export script return a dict.
        scores, embeddings = out["predictions"], out["embeddings"]
    else:
        # The published google/yamnet/1 returns (scores, embeddings, log_mel_spectrogram).
        scores, embeddings = out[0], out[1]
    scores, embeddings = scores.numpy(), embeddings.numpy()
    if scores.ndim != 2 or scores.shape[1] != 521 or embeddings.shape[1] != 1024:
        raise RuntimeError(f"Unexpected YAMNet output shapes {scores.shape}, {embeddings.shape}.")
    return scores, embeddings

Writing encore_audio/yamnet_model.py


### `encore_audio/descriptors.py`: measuring the sound

19 measurements of the sound itself: length, attack, decay, brightness, low and high energy, noisiness, pitch (YIN algorithm), number of attacks, sweep. Uses only numpy and scipy.

In [5]:
%%writefile encore_audio/descriptors.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Acoustic descriptors: plain measurements of how a sound behaves.

These numbers describe the SOUND, not the object that made it: how long it is, how fast it
starts, where its energy sits in frequency, whether it has a pitch, whether it changes over
time. Musical roles are largely defined in exactly these terms ("deep thump", "short crisp
high hit", "sustained tone"), which is why they work on objects nobody has recorded before.

Only numpy and scipy are used, so there is nothing exotic to install.
"""
from __future__ import annotations

import numpy as np
from scipy.signal import find_peaks, stft

from . import config as C
from .audio_io import frame_rms

# Fixed order of the numeric vector fed to the learned role model.
DESCRIPTOR_VECTOR_NAMES = [
    "log10_effective_duration_s", "log10_event_duration_s", "log10_attack_ms", "log10_decay_ms", "sustain_ratio",
    "temporal_centroid", "log1p_onset_count", "onset_rate_hz", "log2_centroid_hz",
    "log2_rolloff_hz", "low_ratio", "high_ratio", "flatness", "centroid_std_oct",
    "centroid_drift_oct", "voiced_fraction", "pitch_clarity", "log2_f0_hz", "f0_std_semitones",
]


# ----------------------------------------------------------------------------- envelope
def _envelope_features(x: np.ndarray, sr: int) -> dict:
    """Loudness-over-time shape, from an RMS envelope with 5 ms windows every 1 ms."""
    hop = sr // 1000
    env = frame_rms(x, 5 * hop, hop)
    peak_i = int(np.argmax(env))
    peak = float(env[peak_i]) + 1e-12
    lo, hi = 0.1 * peak, 0.9 * peak

    # Attack: time for the loudest event to rise from 10 % to 90 % of its peak level.
    below = np.flatnonzero(env[: peak_i + 1] < lo)
    t10 = int(below[-1]) + 1 if below.size else 0
    above = np.flatnonzero(env[t10 : peak_i + 1] >= hi)
    t90 = t10 + (int(above[0]) if above.size else peak_i - t10)
    attack_ms = float(max(t90 - t10, 1))

    # Decay: time from the peak until the level first falls below 10 % of the peak.
    after = np.flatnonzero(env[peak_i:] < lo)
    decay_ms = float(max(int(after[0]) if after.size else env.size - peak_i, 1))

    times = np.arange(env.size)
    return {
        "effective_duration_s": float(np.sum(env >= lo)) / 1000.0,  # time within 20 dB of peak
        # Length of the loudest single event: from where it rises above 10 % of its peak to
        # where it falls back below. Three knocks in one recording have a long effective
        # duration but a short event duration, and it is the single knock that would be
        # used as a drum hit.
        "event_duration_s": float(peak_i - t10 + decay_ms) / 1000.0,
        "attack_ms": attack_ms,
        "decay_ms": decay_ms,
        # mean level / peak level. A single decaying hit is ~0.1-0.25, a held tone ~0.6-0.9.
        "sustain_ratio": float(np.mean(env) / peak),
        # where the energy sits in time: 0 = all at the start, 0.5 = evenly spread
        "temporal_centroid": float(np.sum(times * env) / (np.sum(env) + 1e-12) / max(env.size, 1)),
    }


def _onset_features(x: np.ndarray, sr: int) -> dict:
    """Count distinct attacks: places where the level jumps by 6 dB or more within 30 ms."""
    hop = sr // 100
    env_db = 20.0 * np.log10(frame_rms(x, 2 * hop, hop) + 1e-9)
    top = float(env_db.max())
    padded = np.concatenate((np.full(3, top - 60.0), env_db))
    recent_min = np.minimum(np.minimum(padded[:-3], padded[1:-2]), padded[2:-1])
    rise = env_db - recent_min
    peaks, _ = find_peaks(rise, height=6.0, distance=5)  # at least 50 ms apart
    # the level must actually get reasonably loud shortly after the jump
    ahead = np.array([env_db[p : p + 4].max() for p in peaks]) if peaks.size else np.array([])
    count = int(np.sum(ahead >= top - 25.0))
    return {"onset_count": float(count), "onset_rate_hz": count / max(x.size / sr, 0.05)}


# ----------------------------------------------------------------------------- spectrum
def _spectral_features(x: np.ndarray, sr: int) -> dict:
    n_fft, hop = 512, sr // 100  # 32 ms windows every 10 ms
    if x.size < n_fft:
        x = np.pad(x, (0, n_fft - x.size))
    freqs, _, Z = stft(x, fs=sr, window="hann", nperseg=n_fft, noverlap=n_fft - hop,
                       boundary=None, padded=False)
    P = np.abs(Z) ** 2                       # [freq, time] power
    keep_f = freqs >= 50.0                   # ignore DC / handling rumble
    freqs, P = freqs[keep_f], P[keep_f]
    frame_energy = P.sum(axis=0)
    loud = frame_energy >= frame_energy.max() * 1e-3   # frames within 30 dB of the loudest
    P, frame_energy = P[:, loud], frame_energy[loud]
    S = P.sum(axis=1)                        # long-term spectrum of the loud frames
    total = float(S.sum()) + 1e-20

    centroid = float(np.sum(freqs * S) / total)
    rolloff = float(freqs[min(np.searchsorted(np.cumsum(S), 0.85 * total), freqs.size - 1)])

    # Flatness: ~0 for a pure tone, ~0.5 for white noise. Per frame, energy-weighted.
    band = (freqs >= 100.0) & (freqs <= 7500.0)
    Pb = P[band] + 1e-12
    flat_t = np.exp(np.mean(np.log(Pb), axis=0)) / np.mean(Pb, axis=0)
    flatness = float(np.sum(flat_t * frame_energy) / (frame_energy.sum() + 1e-20))

    # Does the spectrum move over time (a sweep, a zipper)? Track the per-frame centroid.
    cent_t = np.log2(np.sum(freqs[:, None] * P, axis=0) / (frame_energy + 1e-20) + 1e-9)
    w = frame_energy / (frame_energy.sum() + 1e-20)
    mean_c = float(np.sum(w * cent_t))
    centroid_std = float(np.sqrt(np.sum(w * (cent_t - mean_c) ** 2)))
    drift = 0.0
    t_idx = np.flatnonzero(loud).astype(np.float64)
    if t_idx.size >= 5:
        t = t_idx * hop / sr
        t_mean = float(np.sum(w * t))
        var_t = float(np.sum(w * (t - t_mean) ** 2))
        if var_t > 1e-9:
            slope = float(np.sum(w * (t - t_mean) * (cent_t - mean_c)) / var_t)  # octaves / s
            drift = slope * float(t[-1] - t[0])                                  # octaves total

    return {
        "centroid_hz": centroid,
        "rolloff_hz": rolloff,
        "low_ratio": float(S[freqs < 250.0].sum() / total),
        "high_ratio": float(S[freqs > 3000.0].sum() / total),
        "flatness": flatness,
        "centroid_std_oct": centroid_std,
        "centroid_drift_oct": drift,
    }


# ----------------------------------------------------------------------------- pitch
def _pitch_features(x: np.ndarray, sr: int, fmin: float = 60.0, fmax: float = 4000.0) -> dict:
    """Pitch tracking with the YIN algorithm (de Cheveigne & Kawahara, 2002).

    For each 64 ms frame we ask: is there a lag tau such that the frame looks like a copy of
    itself shifted by tau? If so the sound is periodic with pitch sr / tau.
    """
    frame, hop = 1024, 256
    tau_min, tau_max = int(sr / fmax), int(sr / fmin)
    win = frame - tau_max
    if x.size < frame:
        x = np.pad(x, (0, frame - x.size))
    x = x.astype(np.float64)
    starts = np.arange(0, x.size - frame + 1, hop)
    frames = np.stack([x[s : s + frame] for s in starts])
    rms = np.sqrt(np.mean(frames[:, :win] ** 2, axis=1))
    use = rms >= 0.1 * rms.max()
    frames = frames[use]

    # YIN difference function d(tau) and its cumulative-mean-normalised form d'(tau).
    base = frames[:, :win]
    d = np.zeros((frames.shape[0], tau_max + 1))
    for tau in range(1, tau_max + 1):
        d[:, tau] = np.sum((base - frames[:, tau : tau + win]) ** 2, axis=1)
    cmnd = np.ones_like(d)
    running = np.cumsum(d[:, 1:], axis=1)
    cmnd[:, 1:] = d[:, 1:] * np.arange(1, tau_max + 1) / (running + 1e-12)

    f0, clarity = [], []
    for row in cmnd:
        seg = row[tau_min : tau_max + 1]
        cand = np.flatnonzero(seg < 0.15)
        if cand.size:                      # first dip below threshold, walked down to its minimum
            i = int(cand[0])
            while i + 1 < seg.size and seg[i + 1] < seg[i]:
                i += 1
        else:
            i = int(np.argmin(seg))
        tau = float(i + tau_min)
        if 0 < i < seg.size - 1:           # parabolic interpolation for sub-sample accuracy
            a, b, c = seg[i - 1], seg[i], seg[i + 1]
            denom = a - 2 * b + c
            if abs(denom) > 1e-12:
                tau += 0.5 * (a - c) / denom
        clarity.append(1.0 - float(seg[i]))
        f0.append(sr / tau)
    f0, clarity = np.array(f0), np.clip(np.array(clarity), 0.0, 1.0)
    voiced = clarity >= 0.85               # i.e. d' < 0.15

    out = {
        "voiced_fraction": float(np.mean(voiced)) if voiced.size else 0.0,
        "pitch_clarity": float(np.median(clarity)) if clarity.size else 0.0,
        "f0_hz": 0.0,
        "f0_std_semitones": 0.0,
    }
    if np.any(voiced):
        semis = 12.0 * np.log2(f0[voiced])
        out["f0_hz"] = float(2.0 ** (np.median(semis) / 12.0))
        out["f0_std_semitones"] = float(np.std(semis))
    return out


# ----------------------------------------------------------------------------- public API
def compute_descriptors(active: np.ndarray, sr: int = C.SAMPLE_RATE) -> dict:
    """All descriptors for one trimmed, normalised waveform. Values are plain floats."""
    x = np.asarray(active, dtype=np.float64)
    d = {"duration_s": x.size / sr}
    d.update(_envelope_features(x, sr))
    d.update(_onset_features(x, sr))
    d.update(_spectral_features(x, sr))
    d.update(_pitch_features(x, sr))
    return {k: float(v) for k, v in d.items()}


def descriptor_vector(d: dict) -> np.ndarray:
    """Descriptors as a fixed-order numeric vector (see DESCRIPTOR_VECTOR_NAMES).
    Durations and frequencies are log-scaled because that is how we hear them."""
    f0 = d["f0_hz"]
    return np.array([
        np.log10(max(d["effective_duration_s"], 1e-3)),
        np.log10(max(d["event_duration_s"], 1e-3)),
        np.log10(max(d["attack_ms"], 1.0)),
        np.log10(max(d["decay_ms"], 1.0)),
        d["sustain_ratio"],
        d["temporal_centroid"],
        np.log1p(d["onset_count"]),
        d["onset_rate_hz"],
        np.log2(max(d["centroid_hz"], 20.0)),
        np.log2(max(d["rolloff_hz"], 20.0)),
        d["low_ratio"],
        d["high_ratio"],
        d["flatness"],
        d["centroid_std_oct"],
        d["centroid_drift_oct"],
        d["voiced_fraction"],
        d["pitch_clarity"],
        np.log2(f0) if f0 > 0 else 0.0,
        d["f0_std_semitones"],
    ], dtype=np.float32)

Writing encore_audio/descriptors.py


### `encore_audio/role_prior.py`: role scores from rules

The acoustic prior. Turns the measurements into fuzzy traits (hit, short, dark, pitched, ...) and the traits into a score for each of the five roles. Hand-written, not trained, and not yet validated.

In [6]:
%%writefile encore_audio/role_prior.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Acoustic prior: role scores computed directly from the acoustic descriptors.

This is the role scorer that works on day one, before any human labels exist. It is NOT a
trained model. It turns each role's definition into a few fuzzy conditions on measured
properties of the sound, for example:

    Kick / Low Hit   = a single hit  AND short  AND dark (low-frequency)
    Hi-Hat / Tick    = a single hit  AND very short  AND bright
    Pad / Drone      = pitched  AND long  AND not a hit  AND steady

It never looks at WHAT made the sound, so it applies to any recording. Its weakness is that
the thresholds below are engineering judgement. They were chosen from the role definitions
and from looking at descriptor ranges on real recordings, and they have not been validated
against human role labels. `scripts/evaluate_roles.py` measures exactly that once labels
exist, and the learned model only replaces this prior if it beats it on unseen categories.
"""
from __future__ import annotations

import math

from . import config as C


def ramp(x: float, lo: float, hi: float) -> float:
    """0 below `lo`, 1 above `hi`, straight line in between."""
    if hi == lo:
        return float(x >= hi)
    return float(min(max((x - lo) / (hi - lo), 0.0), 1.0))


def _log2(x: float) -> float:
    return math.log2(max(x, 1.0))


def compute_traits(d: dict) -> dict:
    """Fuzzy yes/no properties of the sound, each between 0 and 1."""
    eff = d["effective_duration_s"]   # all the time the recording is "on"
    event = d["event_duration_s"]     # the loudest single event only
    lc = _log2(d["centroid_hz"])

    # a "hit": energy concentrated in a fast-rising, quickly decaying burst
    hit = (1 - ramp(d["sustain_ratio"], 0.15, 0.35)) * (1 - ramp(d["attack_ms"], 20, 80))
    pitched = max(ramp(d["voiced_fraction"], 0.15, 0.5), 0.7 * ramp(d["pitch_clarity"], 0.55, 0.85))
    dark = max(1 - ramp(lc, _log2(300), _log2(800)), ramp(d["low_ratio"], 0.35, 0.7))
    bright = max(ramp(lc, _log2(1800), _log2(3500)), ramp(d["high_ratio"], 0.15, 0.5))
    bright = min(bright, 1 - dark)
    f0 = d["f0_hz"]
    low_pitch = pitched * (1 - ramp(_log2(f0), _log2(130), _log2(260))) if f0 > 0 else 0.0
    high_pitch = pitched * ramp(_log2(f0), _log2(130), _log2(260)) if f0 > 0 else 0.0
    moving = ramp(abs(d["centroid_drift_oct"]), 0.5, 1.5) * ramp(eff, 0.25, 0.6)

    return {
        "hit": hit,
        # Hit-type roles are judged on the single loudest event, so that knocking three
        # times instead of once does not change the answer.
        "short": 1 - ramp(event, 0.15, 0.5),
        "very_short": 1 - ramp(event, 0.08, 0.3),
        "long": ramp(eff, 0.4, 1.2),
        "pitched": pitched,
        "unpitched": 1 - pitched,
        "dark": dark,
        "bright": bright,
        "busy": ramp(d["onset_count"], 2, 6),          # several separate attacks
        "moving": moving,                              # spectrum sweeps up or down
        "steady": (1 - moving) * (1 - ramp(d["f0_std_semitones"], 1.0, 4.0)),
        "low_pitch": low_pitch,
        "high_pitch": high_pitch,
        "long_tail": (1 - ramp(d["attack_ms"], 20, 80)) * ramp(d["decay_ms"], 300, 1000),
    }


# A role that fails one of its conditions still gets up to this much "partial credit" for the
# conditions it does meet. Without it, every role after the first one or two would score
# exactly 0 on about half of all recordings, and the third suggestion would be an arbitrary
# pick. With it, the remaining places go to the roles that are acoustically closest. The value
# is below config.WEAK_ROLE_SCORE on purpose, so a suggestion that only exists because of
# partial credit is always reported as "weak".
PARTIAL_CREDIT = 0.25


def _mean(values: list) -> float:
    return float(sum(values) / len(values))


def prior_scores(d: dict) -> dict:
    """Role fit scores in [0, 1] for all five roles. These are heuristic fit scores,
    not probabilities.

    score = max(strict, PARTIAL_CREDIT * partial)
      strict   all of the role's conditions multiplied together (every one must hold)
      partial  the average of the same conditions (how many of them hold)
    """
    t = compute_traits(d)
    eff = d["effective_duration_s"]
    event = d["event_duration_s"]
    sweeping = ramp(abs(d["centroid_drift_oct"]), 0.5, 1.5)
    grainy = t["busy"] * (1 - ramp(d["sustain_ratio"], 0.2, 0.4))

    strict = {
        # hits, impacts, thumps, claps -- any single-event percussive sound
        "percussion": t["hit"] * (1 - ramp(event, 0.5, 1.0)),
        # repeating patterns: ticking, rattling, shaking, jingling
        "rhythmic": max(0.9 * grainy, 0.7 * t["busy"]) * (1 - t["long"]),
        # pitched or tonal: hums, whistles, bass, anything with a note
        "melodic": t["pitched"] * (0.5 + 0.5 * ramp(eff, 0.05, 0.3)),
        # sustained backgrounds, drones, washes, atmosphere
        "ambient": t["long"] * (1 - t["hit"]) * (0.5 + 0.5 * t["steady"]),
        # effects, sweeps, bursts, one-shot accents
        "transition": max(t["moving"], 0.7 * t["long_tail"],
                         0.5 * t["hit"] * ramp(d["decay_ms"], 300, 1000)),
    }
    partial = {
        "percussion": _mean([t["hit"], 1 - ramp(event, 0.5, 1.0)]),
        "rhythmic": _mean([t["busy"], 1 - t["long"], 1 - t["hit"]]),
        "melodic": _mean([t["pitched"], ramp(eff, 0.05, 0.3)]),
        "ambient": _mean([t["long"], 1 - t["hit"], t["steady"]]),
        "transition": max(_mean([sweeping, ramp(eff, 0.25, 0.6)]),
                         _mean([1 - ramp(d["attack_ms"], 20, 80), ramp(d["decay_ms"], 300, 1000)])),
    }
    return {role: float(min(max(strict[role], PARTIAL_CREDIT * partial[role], 0.0), 1.0))
            for role in C.ROLE_IDS}


def describe_character(d: dict) -> list[str]:
    """A few plain-language words about the sound, for display and for explaining results."""
    t = compute_traits(d)
    words = []
    if t["hit"] > 0.6:
        words.append("single hit" if t["busy"] < 0.5 else "series of hits")
    elif t["busy"] > 0.5:
        words.append("many small attacks")
    if t["long"] > 0.6:
        words.append("sustained")
    elif t["short"] > 0.6:
        words.append("short")
    if t["dark"] > 0.6:
        words.append("low / dark")
    elif t["bright"] > 0.6:
        words.append("bright")
    else:
        words.append("mid-range")
    if t["pitched"] > 0.6:
        words.append(f"pitched (about {d['f0_hz']:.0f} Hz)" if d["f0_hz"] > 0 else "tonal")
    elif t["pitched"] < 0.3:
        words.append("unpitched / noisy")
    if t["moving"] > 0.6:
        words.append("sweeping")
    if t["long_tail"] > 0.6:
        words.append("long ring-out")
    return words

Writing encore_audio/role_prior.py


### `encore_audio/role_model.py`: the learned role model (inference side)

File format, loading and prediction for the trained model: five logistic regressions stored as plain numpy arrays. Prediction is three lines of numpy, so the backend does not need scikit-learn.

In [7]:
%%writefile encore_audio/role_model.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""The learned role model: file format, feature vector, loading and prediction.

The model is ten independent logistic regressions (one "is this role suitable?" question
per role) on top of frozen YAMNet embeddings plus the acoustic descriptors. That is small
enough to store as plain arrays:

    models/role_model.npz    mean, scale, coef, intercept     (numpy, no pickle)
    models/role_model.json   which roles, which features, versions, training summary

Prediction is three lines of numpy, so the backend needs neither scikit-learn nor the
training data:

    z = (x - mean) / scale
    logits = z @ coef.T + intercept
    probability = 1 / (1 + exp(-logits))
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

from . import config as C
from .descriptors import DESCRIPTOR_VECTOR_NAMES, descriptor_vector

FEATURE_SETS = ("emb", "desc", "emb+desc")
FORMAT_VERSION = 1


def build_feature_vector(embedding_mean: np.ndarray, descriptors: dict, feature_set: str) -> np.ndarray:
    """The model input for one clip.

    embedding_mean: YAMNet's 1024-d embeddings averaged over the clip's frames.
    """
    if feature_set == "emb":
        return np.asarray(embedding_mean, dtype=np.float32)
    if feature_set == "desc":
        return descriptor_vector(descriptors)
    if feature_set == "emb+desc":
        return np.concatenate([np.asarray(embedding_mean, dtype=np.float32), descriptor_vector(descriptors)])
    raise ValueError(f"Unknown feature set {feature_set!r}. Use one of {FEATURE_SETS}.")


def feature_dim(feature_set: str) -> int:
    return {"emb": 1024, "desc": len(DESCRIPTOR_VECTOR_NAMES),
            "emb+desc": 1024 + len(DESCRIPTOR_VECTOR_NAMES)}[feature_set]


class RoleModel:
    def __init__(self, mean, scale, coef, intercept, meta: dict):
        self.mean, self.scale, self.coef, self.intercept = mean, scale, coef, intercept
        self.meta = meta
        self.role_ids: list[str] = meta["role_ids"]       # roles this model can score
        self.feature_set: str = meta["feature_set"]

    def predict(self, embedding_mean: np.ndarray, descriptors: dict) -> dict:
        """Probability-like score per covered role."""
        x = build_feature_vector(embedding_mean, descriptors, self.feature_set).astype(np.float64)
        z = (x - self.mean) / self.scale
        logits = z @ self.coef.T + self.intercept
        probs = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
        return {role: float(p) for role, p in zip(self.role_ids, probs)}


def save_role_model(mean, scale, coef, intercept, meta: dict,
                    npz_path: Path = None, json_path: Path = None) -> None:
    npz_path = Path(npz_path or C.ROLE_MODEL_NPZ)
    json_path = Path(json_path or C.ROLE_MODEL_JSON)
    npz_path.parent.mkdir(parents=True, exist_ok=True)
    meta = dict(meta, format_version=FORMAT_VERSION, preprocess_version=C.PREPROCESS_VERSION,
                encore_version=C.VERSION, all_role_ids=C.ROLE_IDS)
    np.savez(npz_path, mean=np.asarray(mean, np.float64), scale=np.asarray(scale, np.float64),
             coef=np.asarray(coef, np.float64), intercept=np.asarray(intercept, np.float64))
    json_path.write_text(json.dumps(meta, indent=2))


_cache: dict = {}


def load_role_model(npz_path: Path = None, json_path: Path = None):
    """Returns (RoleModel or None, reason). None means "use the acoustic prior"; `reason`
    says why, so the output can be honest about which method produced the suggestions."""
    npz_path = Path(npz_path or C.ROLE_MODEL_NPZ)
    json_path = Path(json_path or C.ROLE_MODEL_JSON)
    if not (npz_path.is_file() and json_path.is_file()):
        return None, "no trained role model found"
    key = (str(npz_path), npz_path.stat().st_mtime_ns, json_path.stat().st_mtime_ns)
    if key in _cache:
        return _cache[key]
    try:
        meta = json.loads(json_path.read_text())
        if meta.get("format_version") != FORMAT_VERSION:
            result = (None, "role model file format is not supported by this code version")
        elif meta.get("preprocess_version") != C.PREPROCESS_VERSION:
            result = (None, "role model was trained with different preprocessing; retrain it")
        elif not set(meta["role_ids"]) <= set(C.ROLE_IDS):
            result = (None, "role model contains unknown roles")
        else:
            with np.load(npz_path, allow_pickle=False) as z:
                arrays = [z["mean"], z["scale"], z["coef"], z["intercept"]]
            dim = feature_dim(meta["feature_set"])
            if arrays[0].shape != (dim,) or arrays[2].shape != (len(meta["role_ids"]), dim):
                result = (None, "role model arrays have the wrong shape")
            else:
                result = (RoleModel(*arrays, meta), "ok")
    except Exception as exc:
        result = (None, f"role model could not be loaded ({type(exc).__name__}: {exc})")
    _cache.clear()
    _cache[key] = result
    return result

Writing encore_audio/role_model.py


### `encore_audio/classify.py`: the public function

`classify_audio_file(path)` ties the pieces together and builds the JSON result, including the review flags and which method produced the roles.

In [8]:
%%writefile encore_audio/classify.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""The public entry point: classify_audio_file(audio_path) -> JSON-serialisable dict.

Pipeline for one file:

    load_audio()            decode, validate, trim, normalise           (audio_io.py)
    run_yamnet()            521 class scores + 1024-d embedding / frame (yamnet_model.py)
    compute_descriptors()   duration, attack, brightness, pitch, ...    (descriptors.py)
    role scores             learned model if one is installed,          (role_model.py)
                            otherwise the acoustic prior                (role_prior.py)

`analyze_audio_file` does the first three steps and is also what the training scripts call,
so training and inference cannot drift apart.
"""
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from . import config as C
from .audio_io import AudioRejected, PreparedAudio, load_audio
from .descriptors import compute_descriptors
from .role_model import load_role_model
from .role_prior import describe_character, prior_scores
from .yamnet_model import YAMNET_HANDLE, class_names, load_yamnet, run_yamnet

FLAG_MESSAGES = {
    "low_label_confidence": "YAMNet is not confident about any sound label. Treat the labels as guesses.",
    "weak_role_evidence": "No role fits this sound clearly. The suggestions are weak guesses.",
    "clipped": "The recording is clipped (too loud for the microphone). Results may be off.",
    "steady_or_noisy": "Little difference between loud and quiet parts: either a continuous sound "
                       "or strong background noise. The two cannot be told apart from one recording.",
    "truncated": f"Only the first {C.MAX_ANALYSIS_SECONDS:.0f} seconds were analysed.",
    "short_for_yamnet": "The sound is much shorter than YAMNet's 0.96 s window, so sound labels "
                        "are less reliable. Role suggestions are less affected.",
}
REVIEW_FLAGS = {"low_label_confidence", "weak_role_evidence", "clipped", "steady_or_noisy"}


@dataclass
class Analysis:
    prepared: PreparedAudio
    scores_mean: np.ndarray      # [521] YAMNet class scores averaged over frames
    embedding_mean: np.ndarray   # [1024] YAMNet embeddings averaged over frames
    n_frames: int
    descriptors: dict


def warm_up() -> None:
    """Load YAMNet now (takes a few seconds). Call once at backend start-up so the first
    user request is not slow."""
    load_yamnet()


def analyze_audio_file(audio_path: str | Path) -> Analysis:
    """Shared front half of the pipeline. Raises AudioRejected for unusable files."""
    prepared = load_audio(audio_path)
    scores, embeddings = run_yamnet(prepared.yamnet_input)
    return Analysis(
        prepared=prepared,
        scores_mean=scores.mean(axis=0),
        embedding_mean=embeddings.mean(axis=0),
        n_frames=int(scores.shape[0]),
        descriptors=compute_descriptors(prepared.active),
    )


def score_roles(embedding_mean: np.ndarray, descriptors: dict, method: str | None = None):
    """Scores for all five roles.

    Returns (scores, sources, method_info). When a trained role model is installed it scores
    the roles it was trained on; any role it could not be trained on (too few labelled
    examples) is scored by the acoustic prior, and `sources` says which is which.
    """
    method = method or os.environ.get("ENCORE_ROLE_METHOD", "auto")
    scores = prior_scores(descriptors)
    sources = {r: "acoustic_prior" for r in C.ROLE_IDS}
    info = {"roles": "acoustic-prior-v0", "role_model": None,
            "role_score_meaning": "heuristic fit score in [0, 1], not a probability"}
    if method == "prior":
        info["role_model_note"] = "learned model disabled (ENCORE_ROLE_METHOD=prior)"
        return scores, sources, info
    model, reason = load_role_model()
    if model is None:
        info["role_model_note"] = reason
        return scores, sources, info
    for role, p in model.predict(embedding_mean, descriptors).items():
        scores[role], sources[role] = p, "learned"
    meta = model.meta
    info.update(
        roles="learned-logreg-v1" + (" + acoustic-prior fallback" if len(model.role_ids) < len(C.ROLE_IDS) else ""),
        role_model={k: meta.get(k) for k in ("trained_at", "feature_set", "role_ids", "n_clips",
                                             "n_groups", "n_raters", "evaluation")},
        role_score_meaning="learned roles: model score in [0, 1] (uncalibrated); "
                           "acoustic_prior roles: heuristic fit score",
    )
    return scores, sources, info


def _strength(score: float) -> str:
    """Plain-word reading of a role score, so the UI does not have to invent thresholds."""
    if score >= C.STRONG_ROLE_SCORE:
        return "strong"
    return "moderate" if score >= C.WEAK_ROLE_SCORE else "weak"


def _roles_block(scores: dict | None, sources: dict | None, top_k: int):
    order = sorted(C.ROLE_IDS, key=lambda r: -scores[r]) if scores else []
    suggested = order[:top_k]
    all_roles = [{
        "role_id": r, "role": C.ROLE_NAMES[r], "description": C.ROLE_DESCRIPTIONS[r],
        "score": round(scores[r], 4) if scores else None,
        "strength": _strength(scores[r]) if scores else None,
        "source": sources[r] if sources else None,
        "suggested": r in suggested,
    } for r in C.ROLE_IDS]
    suggestions = [{"rank": i + 1, "role_id": r, "role": C.ROLE_NAMES[r],
                    "score": round(scores[r], 4), "strength": _strength(scores[r]),
                    "source": sources[r]}
                   for i, r in enumerate(suggested)]
    return suggestions, all_roles


def _method_block(extra: dict | None = None) -> dict:
    block = {
        "encore_version": C.VERSION,
        "preprocess_version": C.PREPROCESS_VERSION,
        "sound_labels": f"yamnet ({YAMNET_HANDLE}), mean class score over 0.96 s frames",
        "sound_label_score_meaning": "YAMNet sigmoid score in [0, 1] per class; scores do not sum to 1",
    }
    block.update(extra or {})
    return block


def classify_audio_file(audio_path: str | Path, top_k_labels: int = 5, top_k_roles: int = 2) -> dict:
    """Classify one audio file. Always returns a JSON-serialisable dict and never raises for
    a bad recording: check result["status"].

    status == "ok"        labels and roles are filled in; see result["review"] for caveats
    status == "rejected"  the file could not be analysed; see result["error"]. `all_roles`
                          is still present so the UI can let the user pick a role by hand.
    """
    try:
        analysis = analyze_audio_file(audio_path)
    except AudioRejected as rej:
        _, all_roles = _roles_block(None, None, top_k_roles)
        return {
            "status": "rejected",
            "error": {"code": rej.code, "message": rej.message},
            "sound_labels": [], "role_suggestions": [], "all_roles": all_roles,
            "sound_character": [],
            "review": {"needs_review": True, "flags": [rej.code], "messages": [rej.message]},
            "audio": None, "descriptors": None,
            "method": _method_block(),
        }

    names = class_names()
    top = np.argsort(analysis.scores_mean)[::-1][:top_k_labels]
    sound_labels = [{"rank": i + 1, "label": names[int(k)], "class_index": int(k),
                     "score": round(float(analysis.scores_mean[k]), 4)} for i, k in enumerate(top)]

    scores, sources, role_info = score_roles(analysis.embedding_mean, analysis.descriptors)
    suggestions, all_roles = _roles_block(scores, sources, top_k_roles)

    flags = list(analysis.prepared.flags)
    if sound_labels[0]["score"] < C.LOW_LABEL_SCORE:
        flags.append("low_label_confidence")
    if suggestions[0]["score"] < C.WEAK_ROLE_SCORE:
        flags.append("weak_role_evidence")
    if analysis.prepared.info["analysed_duration_s"] < 0.5:
        flags.append("short_for_yamnet")

    audio = dict(analysis.prepared.info, yamnet_frames=analysis.n_frames)
    return {
        "status": "ok",
        "error": None,
        "sound_labels": sound_labels,
        "role_suggestions": suggestions,
        "all_roles": all_roles,
        "sound_character": describe_character(analysis.descriptors),
        "review": {
            "needs_review": bool(REVIEW_FLAGS & set(flags)),
            "flags": flags,
            "messages": [FLAG_MESSAGES[f] for f in flags],
        },
        "audio": audio,
        "descriptors": {k: round(v, 4) for k, v in analysis.descriptors.items()},
        "method": _method_block(role_info),
    }

Writing encore_audio/classify.py


### `encore_audio/__init__.py`: package entry point

Makes `from encore_audio import classify_audio_file, warm_up` work.

In [9]:
%%writefile encore_audio/__init__.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Encore audio classification: YAMNet sound labels + musical-role suggestions.

Backend usage:

    from encore_audio import classify_audio_file, warm_up
    warm_up()                                   # once, at start-up
    result = classify_audio_file("clip.m4a")    # JSON-serialisable dict
"""
from .classify import analyze_audio_file, classify_audio_file, warm_up
from .config import ROLE_IDS, ROLES, VERSION

__all__ = ["classify_audio_file", "analyze_audio_file", "warm_up", "ROLES", "ROLE_IDS", "VERSION"]
__version__ = VERSION

Writing encore_audio/__init__.py


### `encore_audio/__main__.py`: command line

Lets you run `python -m encore_audio some_sound.m4a` in a terminal.

In [10]:
%%writefile encore_audio/__main__.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Command line use:  python -m encore_audio path/to/clip.m4a [more files ...]"""
import json
import os
import sys

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")  # hide TensorFlow's startup chatter

from .classify import classify_audio_file  # noqa: E402


def main() -> int:
    if len(sys.argv) < 2:
        print(__doc__)
        return 2
    for path in sys.argv[1:]:
        print(json.dumps(classify_audio_file(path), indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing encore_audio/__main__.py


## A2. Training-side code

Not needed by the backend. Used by Part B and by the labeling tool.

### `encore_audio/dataset.py`: manifest and labels

Reads and writes the clip list (`data/manifest.csv`) and the human role labels (`data/labels/labels_<name>.csv`), combines several raters into one target per clip, and measures how much raters agree.

In [11]:
%%writefile encore_audio/dataset.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Training-side bookkeeping: the clip manifest, human labels and the feature file.

Nothing in here is needed at inference time. File formats (all plain CSV so they can be
opened in a spreadsheet and reviewed by a human):

data/manifest.csv            one row per clip
    clip_id    unique id
    path       audio file, relative to the project root
    source     where it came from: "team", "esc50", ...
    category   what made the sound (folder name / dataset class)
    group      LEAKAGE GROUP. Clips with the same group are never split between training
               and testing. Usually equal to category; the same real-world sound from two
               sources (our "clap" and ESC-50 "clapping") shares one group.
    origin     original file name, for traceability

data/labels/labels_<rater>.csv   one row per (clip, rater) decision; the newest row wins
    clip_id, rater, roles (role ids joined by ";"), best_role, unusable (0/1), labeled_at
"""
from __future__ import annotations

import csv
import re
from pathlib import Path

import numpy as np

from . import config as C

DATA_DIR = C.PROJECT_ROOT / "data"
MANIFEST = DATA_DIR / "manifest.csv"
LABELS_DIR = DATA_DIR / "labels"
FEATURES_NPZ = DATA_DIR / "features" / "features.npz"
MANIFEST_FIELDS = ["clip_id", "path", "source", "category", "group", "origin"]
LABEL_FIELDS = ["clip_id", "rater", "roles", "best_role", "unusable", "labeled_at"]

# The same real-world sound under different names -> one leakage group.
GROUP_ALIASES = {
    "clapping": "clap", "claps": "clap",
    "door_wood_knock": "knock", "knocking": "knock",
    "keyboard": "keyboard_typing", "typing": "keyboard_typing",
    "can_opening": "soda_can_opening", "soda_can": "soda_can_opening", "soda": "soda_can_opening",
    "pouring_water": "water", "water_pouring": "water",
    "tongue_click": "tongue_clicking", "tongue": "tongue_clicking",
    "whistle": "whistling", "snapping": "snap", "finger_snap": "snap",
    "key": "keys", "coin": "coins", "zip": "zipper",
}


def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.strip().lower()).strip("_")


def group_for(category: str) -> str:
    slug = slugify(category)
    return GROUP_ALIASES.get(slug, slug)


# ----------------------------------------------------------------------------- manifest
def read_manifest(path: Path = MANIFEST) -> list[dict]:
    if not Path(path).is_file():
        return []
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_manifest(rows: list[dict], path: Path = MANIFEST) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    ids = [r["clip_id"] for r in rows]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate clip_id in manifest.")
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=MANIFEST_FIELDS)
        w.writeheader()
        w.writerows(sorted(rows, key=lambda r: r["clip_id"]))


def merge_into_manifest(new_rows: list[dict]) -> tuple[int, int]:
    """Add or replace rows by clip_id. Returns (rows added, total rows)."""
    rows = {r["clip_id"]: r for r in read_manifest()}
    before = len(rows)
    rows.update({r["clip_id"]: r for r in new_rows})
    write_manifest(list(rows.values()))
    return len(rows) - before, len(rows)


# ----------------------------------------------------------------------------- labels
def read_labels(labels_dir: Path = LABELS_DIR) -> dict:
    """{clip_id: {rater: {"roles": set, "best_role": str, "unusable": bool}}}"""
    out: dict = {}
    for file in sorted(Path(labels_dir).glob("labels_*.csv")):
        with open(file, newline="", encoding="utf-8") as f:
            for row in sorted(csv.DictReader(f), key=lambda r: r["labeled_at"]):
                roles = {r for r in row["roles"].split(";") if r}
                unknown = roles - set(C.ROLE_IDS)
                if unknown:
                    raise ValueError(f"{file.name}: unknown role id(s) {sorted(unknown)}")
                out.setdefault(row["clip_id"], {})[row["rater"]] = {
                    "roles": roles, "best_role": row["best_role"], "unusable": row["unusable"] == "1"}
    return out


def append_label(rater: str, clip_id: str, roles: list[str], best_role: str, unusable: bool,
                 labeled_at: str, labels_dir: Path = LABELS_DIR) -> None:
    Path(labels_dir).mkdir(parents=True, exist_ok=True)
    file = Path(labels_dir) / f"labels_{slugify(rater)}.csv"
    is_new = not file.is_file()
    with open(file, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=LABEL_FIELDS)
        if is_new:
            w.writeheader()
        w.writerow({"clip_id": clip_id, "rater": slugify(rater), "roles": ";".join(roles),
                    "best_role": best_role, "unusable": "1" if unusable else "0", "labeled_at": labeled_at})


def aggregate_labels(labels: dict) -> dict:
    """Combine raters into one target per clip.

    A role counts as ACCEPTABLE for a clip when at least half of the raters who judged the
    clip ticked it. Clips that at least half of their raters marked unusable are dropped.
    Returns {clip_id: {"y": 0/1 array[10], "votes": fraction array[10], "n_raters": int}}.
    """
    out = {}
    for clip_id, by_rater in labels.items():
        n = len(by_rater)
        if sum(v["unusable"] for v in by_rater.values()) * 2 >= n:
            continue
        usable = [v for v in by_rater.values() if not v["unusable"]]
        votes = np.array([sum(r in v["roles"] for v in usable) for r in C.ROLE_IDS], dtype=float) / len(usable)
        y = (votes >= 0.5).astype(int)
        if y.sum() == 0:
            continue
        out[clip_id] = {"y": y, "votes": votes, "n_raters": len(usable)}
    return out


def rater_agreement(labels: dict) -> dict:
    """How much do two humans agree? This is the ceiling any model should be judged against."""
    jaccards, shared = [], []
    for by_rater in labels.values():
        sets = [v["roles"] for v in by_rater.values() if not v["unusable"] and v["roles"]]
        for i in range(len(sets)):
            for j in range(i + 1, len(sets)):
                jaccards.append(len(sets[i] & sets[j]) / len(sets[i] | sets[j]))
                shared.append(bool(sets[i] & sets[j]))
    if not jaccards:
        return {"n_pairs": 0}
    return {"n_pairs": len(jaccards), "mean_jaccard": float(np.mean(jaccards)),
            "share_at_least_one_role": float(np.mean(shared))}


# ----------------------------------------------------------------------------- features
def load_features(path: Path = FEATURES_NPZ) -> dict:
    if not Path(path).is_file():
        raise FileNotFoundError(f"{path} not found. Run: python scripts/extract_features.py")
    with np.load(path, allow_pickle=False) as z:
        feats = {k: z[k] for k in z.files}
    version = str(feats["preprocess_version"])
    if version != C.PREPROCESS_VERSION:
        raise RuntimeError(f"Features were extracted with preprocessing {version}, the code is now "
                           f"{C.PREPROCESS_VERSION}. Re-run scripts/extract_features.py")
    return feats

Writing encore_audio/dataset.py


### `encore_audio/training.py`: fitting and metrics

Fits the five logistic regressions and computes the metrics (hit@2, precision@2, recall@2, top-1, average precision, chance level, intervals). Evaluation and final training both call this, so the model that is evaluated is the model that ships.

In [12]:
%%writefile encore_audio/training.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Fitting and scoring the learned role model. Used by BOTH scripts/evaluate_roles.py and
scripts/train_roles.py, so the thing that gets evaluated is exactly the thing that ships.

Model: for each role, an L2-regularised logistic regression answering "is this role
acceptable for this clip?", on standardised features. Hyperparameters are fixed in advance
(C = 0.1, balanced class weights) and deliberately NOT tuned: with a few hundred clips,
tuning would mostly fit noise and inflate the reported numbers.

A role is only trained when it has at least MIN_POSITIVES acceptable clips (and as many
non-acceptable ones). Roles below that are left to the acoustic prior.

Feature set "emb+desc" is a LATE FUSION: one regression is trained on the 1024 embedding
dimensions and a separate one on the 19 descriptors, and their logits are averaged. Simply
concatenating the two would let 1024 embedding dimensions drown out 19 descriptors under a
shared L2 penalty. Averaging two linear models is itself a linear model on the concatenated
features (each block's weights halved), so the saved file format is the same for all sets.
"""
from __future__ import annotations

from math import comb

import numpy as np

from . import config as C
from .role_model import RoleModel, build_feature_vector  # noqa: F401  (re-exported for scripts)

MIN_POSITIVES = 5
REGULARISATION_C = 0.1


def fit_role_weights(X: np.ndarray, Y: np.ndarray, feature_set: str) -> RoleModel | None:
    """X: [n, d] features, Y: [n, R] 0/1 acceptable roles (columns in C.ROLE_IDS order)."""
    from sklearn.linear_model import LogisticRegression

    n = X.shape[0]
    covered = [j for j in range(len(C.ROLE_IDS)) if MIN_POSITIVES <= Y[:, j].sum() <= n - MIN_POSITIVES]
    if not covered:
        return None
    mean = X.mean(axis=0)
    scale = X.std(axis=0)
    scale[scale < 1e-6] = 1.0
    Z = (X - mean) / scale
    # column blocks that get their own independently trained regression
    blocks = [slice(0, 1024), slice(1024, X.shape[1])] if feature_set == "emb+desc" else [slice(0, X.shape[1])]
    coef = np.zeros((len(covered), X.shape[1]))
    intercept = np.zeros(len(covered))
    for row, j in enumerate(covered):
        for block in blocks:
            lr = LogisticRegression(C=REGULARISATION_C, class_weight="balanced", max_iter=5000)
            lr.fit(Z[:, block], Y[:, j])
            coef[row, block] = lr.coef_[0] / len(blocks)
            intercept[row] += lr.intercept_[0] / len(blocks)
    meta = {"role_ids": [C.ROLE_IDS[j] for j in covered], "feature_set": feature_set}
    return RoleModel(mean, scale, coef, intercept, meta)


def model_scores(model: RoleModel | None, X: np.ndarray, prior: np.ndarray) -> np.ndarray:
    """Scores [n, 10] exactly as inference combines them: learned where the model covers a
    role, acoustic prior elsewhere."""
    scores = prior.astype(np.float64).copy()
    if model is None:
        return scores
    Z = (X - model.mean) / model.scale
    probs = 1.0 / (1.0 + np.exp(-np.clip(Z @ model.coef.T + model.intercept, -30, 30)))
    for k, role in enumerate(model.role_ids):
        scores[:, C.ROLE_IDS.index(role)] = probs[:, k]
    return scores


# ----------------------------------------------------------------------------- metrics
def per_clip_metrics(scores: np.ndarray, Y: np.ndarray, k: int = 2) -> dict:
    """One value per clip for each metric.

    hit@k        1 if at least one of the top-k suggestions is acceptable
    precision@k  fraction of the top-k suggestions that are acceptable
    recall@k     fraction of the acceptable roles found in the top k (capped at k)
    top1         1 if the first suggestion is acceptable
    ap           average precision of the full ranking (1.0 = all acceptable roles on top)
    """
    order = np.argsort(-scores, axis=1, kind="stable")
    ranked = np.take_along_axis(Y, order, axis=1).astype(float)   # acceptability by rank
    n_pos = Y.sum(axis=1)
    topk = ranked[:, :k].sum(axis=1)
    precision_at_rank = np.cumsum(ranked, axis=1) / np.arange(1, Y.shape[1] + 1)
    return {
        f"hit@{k}": (topk > 0).astype(float),
        f"precision@{k}": topk / k,
        f"recall@{k}": topk / np.minimum(n_pos, k),
        "top1": ranked[:, 0],
        "ap": (precision_at_rank * ranked).sum(axis=1) / n_pos,
    }


def chance_metrics(Y: np.ndarray, k: int = 2) -> dict:
    """What a uniformly random ranking would score in expectation, per clip. Essential
    context: if raters tick two roles per clip out of five, random guessing already gets hit@2 = 0.60."""
    r = Y.shape[1]
    n_pos = Y.sum(axis=1)
    hit = np.array([1 - comb(r - int(p), k) / comb(r, k) for p in n_pos])
    return {f"hit@{k}": hit, f"precision@{k}": n_pos / r, f"recall@{k}": (k * n_pos / r) / np.minimum(n_pos, k),
            "top1": n_pos / r, "ap": np.full(len(n_pos), np.nan)}


def cluster_bootstrap_ci(values: np.ndarray, groups: np.ndarray, n_boot: int = 2000, seed: int = 0):
    """Mean and 95 % interval, resampling whole groups (sound categories) rather than clips,
    because clips of one category are not independent."""
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups)
    sums = np.array([values[groups == g].sum() for g in uniq])
    counts = np.array([(groups == g).sum() for g in uniq])
    idx = rng.integers(0, len(uniq), size=(n_boot, len(uniq)))
    boots = sums[idx].sum(axis=1) / counts[idx].sum(axis=1)
    return float(values.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

Writing encore_audio/training.py


## A3. Scripts

Each step of Part B as a small program. Part B runs them for you; they also work from a terminal
(`python scripts/<name>.py`).

### `scripts/download_yamnet.py`: fetch YAMNet

One-time download into `models/yamnet/`.

In [13]:
%%writefile scripts/download_yamnet.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Download Google's pretrained YAMNet (about 17 MB) into models/yamnet/.

    python scripts/download_yamnet.py

You only need this once. classify_audio_file() also does it automatically on first use; the
script exists so a deployment can fetch the model at build time instead of on the first
user request. After the download the project works offline.
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from encore_audio.yamnet_model import download_yamnet, yamnet_is_downloaded  # noqa: E402

if __name__ == "__main__":
    if yamnet_is_downloaded():
        print("YAMNet is already in models/yamnet/. Nothing to do.")
    else:
        download_yamnet()

Writing scripts/download_yamnet.py


### `scripts/self_check.py`: installation check

Pushes synthetic sounds through the whole pipeline. Checks the install, the formats and the rejection of bad files. It is a sanity check, not a measure of accuracy on real recordings.

In [14]:
%%writefile scripts/self_check.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Installation self-check. Run from the project root:

    python scripts/self_check.py

It generates synthetic sounds, pushes them through the full pipeline and checks that:
  * YAMNet is installed correctly (a sine wave is recognised as "Sine wave", etc.),
  * WAV, M4A and MP3 files all decode,
  * silent / corrupt / missing / tiny files are rejected cleanly instead of crashing,
  * the output is JSON-serialisable and has the agreed shape,
  * the acoustic prior puts textbook-simple sounds in the expected role.

IMPORTANT: the role checks are definitional sanity checks on synthetic signals (a 60 Hz thump
should count as a kick). They are NOT evidence of accuracy on real recordings. Real accuracy
is measured by scripts/evaluate_roles.py against human labels.
"""
from __future__ import annotations

import json
import os
import subprocess
import sys
import tempfile
import wave
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import numpy as np  # noqa: E402

from encore_audio import ROLE_IDS, classify_audio_file  # noqa: E402
from encore_audio.audio_io import find_ffmpeg  # noqa: E402

SR = 44_100  # deliberately not 16 kHz, and stereo below, to exercise resampling and downmix
rng = np.random.default_rng(320)


def write_wav(path: Path, x: np.ndarray, sr: int = SR, stereo: bool = True) -> Path:
    pcm = (np.clip(x, -1, 1) * 32767).astype("<i2")
    if stereo:
        pcm = np.repeat(pcm[:, None], 2, axis=1)
    with wave.open(str(path), "wb") as w:
        w.setnchannels(2 if stereo else 1)
        w.setsampwidth(2)
        w.setframerate(sr)
        w.writeframes(pcm.tobytes())
    return path


def t(seconds: float) -> np.ndarray:
    return np.arange(int(seconds * SR)) / SR


def noise(seconds: float) -> np.ndarray:
    return rng.uniform(-1, 1, int(seconds * SR))


def bandpass(x: np.ndarray, lo: float, hi: float) -> np.ndarray:
    spec = np.fft.rfft(x)
    f = np.fft.rfftfreq(x.size, 1 / SR)
    spec[(f < lo) | (f > hi)] = 0
    return np.fft.irfft(spec, x.size)


def with_silence(x: np.ndarray, before: float = 0.3, after: float = 0.5) -> np.ndarray:
    quiet = lambda s: 1e-4 * noise(s)  # a realistic, very quiet noise floor
    out = np.concatenate([quiet(before), x, quiet(after)])
    return 0.5 * out / np.max(np.abs(out))


def make_sounds() -> dict:
    thump = np.sin(2 * np.pi * 60 * t(0.25)) * np.exp(-t(0.25) / 0.05)
    tick = bandpass(noise(0.03), 5000, 12000) * np.exp(-t(0.03) / 0.006)
    crack = bandpass(noise(0.15), 600, 3500) * np.exp(-t(0.15) / 0.03)
    tone = np.sin(2 * np.pi * 440 * t(1.5)) * np.minimum(1, t(1.5) / 0.05)
    low_tone = sum(np.sin(2 * np.pi * 70 * k * t(1.5)) / k for k in (1, 2, 3)) * np.minimum(1, t(1.5) / 0.05)
    hiss = bandpass(noise(3.0), 300, 6000) * np.minimum(1, t(3.0) / 0.3)
    sweep_f = 200 * (4000 / 200) ** (t(1.5) / 1.5)
    sweep = np.sin(2 * np.pi * np.cumsum(sweep_f) / SR)
    rattle = np.zeros(int(1.2 * SR))
    for start in np.sort(rng.uniform(0, 1.1, 14)):
        i = int(start * SR)
        g = bandpass(noise(0.03), 2500, 9000) * np.exp(-t(0.03) / 0.008)
        rattle[i : i + g.size] += g[: rattle.size - i]
    return {
        # name: (waveform, role that must appear in the top two)
        "low_thump": (with_silence(thump), "percussion"),
        "bright_tick": (with_silence(tick), "percussion"),
        "noise_crack": (with_silence(crack), "percussion"),
        "steady_440hz_tone": (with_silence(tone), "melodic"),
        "steady_70hz_tone": (with_silence(low_tone), "ambient"),
        "steady_hiss": (with_silence(hiss), "ambient"),
        "rising_sweep": (with_silence(sweep), "melodic"),
        "rattle": (with_silence(rattle), "rhythmic"),
    }


class Checker:
    def __init__(self):
        self.failed = 0

    def check(self, ok: bool, name: str, detail: str = "") -> None:
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f"  ({detail})" if detail else ""))
        self.failed += 0 if ok else 1


def main() -> int:
    c = Checker()
    tmp = Path(tempfile.mkdtemp(prefix="encore_selfcheck_"))

    print("1. YAMNet itself (same checks as Google's own yamnet_test.py)")
    sine = write_wav(tmp / "sine.wav", 0.5 * np.sin(2 * np.pi * 440 * t(3.0)))
    white = write_wav(tmp / "white.wav", 0.5 * noise(3.0))
    for path, expected in ((sine, "Sine wave"), (white, "White noise")):
        res = classify_audio_file(path, top_k_labels=10)
        labels = [l["label"] for l in res["sound_labels"]]
        c.check(res["status"] == "ok" and expected in labels, f"{path.name} -> '{expected}' in top 10",
                ", ".join(labels[:3]))

    print("2. Output contract")
    res = classify_audio_file(sine)
    json.dumps(res)  # raises if not serialisable
    c.check(True, "result is JSON-serialisable")
    c.check(len(res["sound_labels"]) == 5, "5 sound labels")
    c.check(len(res["role_suggestions"]) == 2, "2 role suggestions")
    c.check([r["role_id"] for r in res["all_roles"]] == ROLE_IDS, "all 5 roles listed in canonical order")
    c.check(all(0.0 <= r["score"] <= 1.0 for r in res["all_roles"]), "role scores are within [0, 1]")

    print("3. Formats")
    ffmpeg = find_ffmpeg()
    for ext, codec in ((".m4a", ["-c:a", "aac"]), (".mp3", ["-c:a", "libmp3lame"])):
        out = tmp / f"sine{ext}"
        enc = subprocess.run([ffmpeg, "-nostdin", "-v", "error", "-y", "-i", str(sine), *codec, str(out)],
                             capture_output=True)
        if enc.returncode != 0:
            print(f"  [SKIP] could not create a test {ext} with this ffmpeg build")
            continue
        r = classify_audio_file(out, top_k_labels=10)
        c.check(r["status"] == "ok" and "Sine wave" in [l["label"] for l in r["sound_labels"]],
                f"{ext} decodes and classifies")

    print("4. Bad input is rejected cleanly")
    silent = write_wav(tmp / "silent.wav", np.zeros(SR))
    garbage = tmp / "garbage.m4a"
    garbage.write_bytes(os.urandom(4096))
    empty = tmp / "empty.wav"
    empty.write_bytes(b"")
    tiny = write_wav(tmp / "tiny.wav", 0.5 * noise(0.01))
    for path, code in ((silent, "silent"), (garbage, "decode_failed"), (empty, "empty"),
                       (tiny, "too_short"), (tmp / "missing.wav", "file_not_found")):
        r = classify_audio_file(path)
        got = (r["error"] or {}).get("code")
        c.check(r["status"] == "rejected" and got == code and len(r["all_roles"]) == 5,
                f"{path.name} -> rejected as '{code}'", f"got {got}")
    long = write_wav(tmp / "long.wav", 0.3 * noise(20.0))
    r = classify_audio_file(long)
    c.check(r["status"] == "ok" and "truncated" in r["review"]["flags"], "20 s file is truncated and flagged")
    click = write_wav(tmp / "click.wav", with_silence(bandpass(noise(0.03), 5000, 12000), 0.02, 0.02))
    r = classify_audio_file(click)
    c.check(r["status"] == "ok" and "short_for_yamnet" in r["review"]["flags"], "very short clip still works, flagged")

    print("5. Acoustic prior on textbook synthetic sounds (sanity only, not accuracy)")
    for name, (wav, expected) in make_sounds().items():
        r = classify_audio_file(write_wav(tmp / f"{name}.wav", wav))
        top2 = [s["role_id"] for s in r["role_suggestions"]]
        c.check(expected in top2, f"{name} -> '{expected}' in top 2", ", ".join(top2))

    print()
    if c.failed:
        print(f"{c.failed} check(s) FAILED.")
        return 1
    print("All checks passed.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing scripts/self_check.py


### `scripts/add_recordings.py`: register our recordings

Adds the files in `data/my_recordings/<category>/` to the manifest.

In [15]:
%%writefile scripts/add_recordings.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Register your own recordings in the manifest.

Put the files in one folder per sound category, for example:

    data/my_recordings/
        bottle/   bottle_1.m4a  bottle_2.m4a  bottle_3.m4a
        clap/     clap_1.m4a ...
        electric_toothbrush/  ...

Then run:

    python scripts/add_recordings.py                       # uses data/my_recordings
    python scripts/add_recordings.py --dir some/other/dir --source mike_phone

The folder name becomes the category. The files stay where they are (M4A, WAV and MP3 are all
fine); only data/manifest.csv is updated. Running it again is safe.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from encore_audio import config as C  # noqa: E402
from encore_audio.dataset import DATA_DIR, group_for, merge_into_manifest, slugify  # noqa: E402

AUDIO_EXTENSIONS = {".m4a", ".wav", ".mp3", ".aac", ".ogg", ".flac", ".mp4", ".webm", ".3gp", ".opus"}


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--dir", type=Path, default=DATA_DIR / "my_recordings")
    ap.add_argument("--source", default="team", help="short name for where these came from")
    args = ap.parse_args()

    root = args.dir.resolve()
    if not root.is_dir():
        print(f"{root} does not exist. Create it with one sub-folder per sound category.")
        return 1
    try:
        root.relative_to(C.PROJECT_ROOT)
    except ValueError:
        print(f"Please keep recordings inside the project folder ({C.PROJECT_ROOT}) so that the "
              "manifest paths work on your teammates' computers too.")
        return 1

    rows = []
    for folder in sorted(p for p in root.iterdir() if p.is_dir()):
        category = slugify(folder.name)
        for file in sorted(folder.iterdir()):
            if file.suffix.lower() not in AUDIO_EXTENSIONS:
                continue
            rows.append({
                "clip_id": f"{slugify(args.source)}__{category}__{slugify(file.stem)}",
                "path": str(file.relative_to(C.PROJECT_ROOT)),
                "source": slugify(args.source), "category": category,
                "group": group_for(category), "origin": file.name,
            })
    if not rows:
        print(f"No audio files found in sub-folders of {root}.")
        return 1
    added, total = merge_into_manifest(rows)
    cats = sorted({r["category"] for r in rows})
    print(f"Registered {len(rows)} recordings in {len(cats)} categories: {', '.join(cats)}")
    print(f"Manifest: {added} new rows, {total} total -> data/manifest.csv")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing scripts/add_recordings.py


### `scripts/prepare_esc50.py`: sample ESC-50

Only needed to draw a different ESC-50 sample. The 300 prepared clips are already in the repo.

In [16]:
%%writefile scripts/prepare_esc50.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Add a sample of ESC-50 clips to the labeling pool.

ESC-50 (Piczak, 2015) is 2,000 five-second recordings of 50 everyday sound classes. It gives
us many sound categories we never recorded ourselves, which is what makes an "unseen
category" evaluation possible. It contains NO musical-role labels: those come from the team
through label_app.py.

Get the dataset once (about 1.4 GB on disk):

    git clone --depth 1 https://github.com/karolpiczak/ESC-50.git data/raw/ESC-50

Then:

    python scripts/prepare_esc50.py --per-class 6

Each selected clip is trimmed and cut to its first 3 seconds of sound, because Encore users
record short sounds and a role judgement depends heavily on length. The cut clip is saved to
data/clips/esc50/ and that exact file is what gets labelled, trained on and evaluated.

License: ESC-50 is CC BY-NC (non-commercial, attribution required). Fine for a course
project; cite it in the report.
"""
from __future__ import annotations

import argparse
import csv
import random
import sys
import wave
from collections import defaultdict
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import numpy as np  # noqa: E402

from encore_audio import config as C  # noqa: E402
from encore_audio.audio_io import AudioRejected, load_audio  # noqa: E402
from encore_audio.dataset import DATA_DIR, group_for, merge_into_manifest  # noqa: E402


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--esc50-dir", type=Path, default=DATA_DIR / "raw" / "ESC-50")
    ap.add_argument("--per-class", type=int, default=6, help="clips to sample from each of the 50 classes")
    ap.add_argument("--max-seconds", type=float, default=3.0)
    ap.add_argument("--seed", type=int, default=320)
    args = ap.parse_args()

    meta = args.esc50_dir / "meta" / "esc50.csv"
    if not meta.is_file():
        print(f"Could not find {meta}.\nClone the dataset first (see the top of this file).")
        return 1
    by_class: dict = defaultdict(list)
    with open(meta, newline="") as f:
        for row in csv.DictReader(f):
            by_class[row["category"]].append(row)

    out_dir = DATA_DIR / "clips" / "esc50"
    out_dir.mkdir(parents=True, exist_ok=True)
    rng = random.Random(args.seed)
    max_len = int(args.max_seconds * C.SAMPLE_RATE)
    rows, skipped = [], 0
    for category in sorted(by_class):
        candidates = sorted(by_class[category], key=lambda r: r["filename"])
        rng.shuffle(candidates)
        used_sources, taken = set(), 0
        for row in candidates:
            if taken == args.per_class:
                break
            if row["src_file"] in used_sources:   # one clip per original Freesound recording
                continue
            try:
                audio = load_audio(args.esc50_dir / "audio" / row["filename"]).active
            except AudioRejected:
                skipped += 1
                continue
            if audio.size > max_len:
                audio = audio[:max_len].copy()
                fade = int(0.02 * C.SAMPLE_RATE)
                audio[-fade:] *= np.linspace(1.0, 0.0, fade, dtype=np.float32)
            clip_id = f"esc50__{category}__{Path(row['filename']).stem}"
            out_path = out_dir / f"{clip_id}.wav"
            with wave.open(str(out_path), "wb") as w:
                w.setnchannels(1)
                w.setsampwidth(2)
                w.setframerate(C.SAMPLE_RATE)
                w.writeframes((np.clip(audio, -1, 1) * 32767).astype("<i2").tobytes())
            rows.append({"clip_id": clip_id, "path": str(out_path.relative_to(C.PROJECT_ROOT)),
                         "source": "esc50", "category": category, "group": group_for(category),
                         "origin": row["filename"]})
            used_sources.add(row["src_file"])
            taken += 1
    added, total = merge_into_manifest(rows)
    print(f"Prepared {len(rows)} ESC-50 clips from {len(by_class)} classes "
          f"({skipped} silent/unreadable clips skipped).")
    print(f"Manifest: {added} new rows, {total} total -> data/manifest.csv")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing scripts/prepare_esc50.py


### `scripts/extract_features.py`: run every clip

Runs every clip in the manifest through the inference pipeline and caches the results.

In [17]:
%%writefile scripts/extract_features.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Run every clip in data/manifest.csv through the SAME front end that inference uses
(encore_audio.classify.analyze_audio_file) and cache the results.

    python scripts/extract_features.py

Writes:
    data/features/features.npz    arrays aligned by row:
        clip_ids [n], embedding_mean [n,1024], desc_vector [n,18], prior_scores [n,10],
        yamnet_scores [n,521], preprocess_version
    data/features/clip_info.csv   human-readable: top YAMNet labels, descriptors, flags,
                                  and any clip that was rejected (with the reason)

Takes roughly 0.1 s per clip on a laptop CPU. No GPU needed.
"""
from __future__ import annotations

import csv
import os
import sys
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import numpy as np  # noqa: E402

from encore_audio import config as C  # noqa: E402
from encore_audio.audio_io import AudioRejected  # noqa: E402
from encore_audio.classify import analyze_audio_file  # noqa: E402
from encore_audio.dataset import FEATURES_NPZ, read_manifest  # noqa: E402
from encore_audio.descriptors import descriptor_vector  # noqa: E402
from encore_audio.role_prior import prior_scores  # noqa: E402
from encore_audio.yamnet_model import class_names  # noqa: E402


def main() -> int:
    manifest = read_manifest()
    if not manifest:
        print("data/manifest.csv is empty. Run scripts/add_recordings.py and/or scripts/prepare_esc50.py first.")
        return 1
    names = class_names()
    ids, emb, desc, prior, yam, info_rows = [], [], [], [], [], []
    for i, row in enumerate(manifest, 1):
        info = {"clip_id": row["clip_id"], "source": row["source"], "category": row["category"]}
        try:
            a = analyze_audio_file(C.PROJECT_ROOT / row["path"])
        except AudioRejected as rej:
            info.update(status="rejected", note=f"{rej.code}: {rej.message}")
            info_rows.append(info)
            continue
        scores = prior_scores(a.descriptors)
        ids.append(row["clip_id"])
        emb.append(a.embedding_mean)
        desc.append(descriptor_vector(a.descriptors))
        prior.append([scores[r] for r in C.ROLE_IDS])
        yam.append(a.scores_mean)
        top = np.argsort(a.scores_mean)[::-1][:5]
        info.update(status="ok", note=";".join(a.prepared.flags),
                    yamnet_top5=" | ".join(f"{names[k]} ({a.scores_mean[k]:.2f})" for k in top),
                    prior_top3=" | ".join(sorted(C.ROLE_IDS, key=lambda r: -scores[r])[:3]),
                    **{k: round(v, 4) for k, v in a.descriptors.items()})
        info_rows.append(info)
        if i % 50 == 0 or i == len(manifest):
            print(f"  {i}/{len(manifest)} clips")

    FEATURES_NPZ.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        FEATURES_NPZ, clip_ids=np.array(ids), embedding_mean=np.array(emb, np.float32),
        desc_vector=np.array(desc, np.float32), prior_scores=np.array(prior, np.float32),
        yamnet_scores=np.array(yam, np.float32), preprocess_version=np.array(C.PREPROCESS_VERSION))
    first = ["clip_id", "source", "category", "status", "note", "yamnet_top5", "prior_top3"]
    fields = first + sorted({k for r in info_rows for k in r} - set(first))
    with open(FEATURES_NPZ.parent / "clip_info.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(info_rows)
    rejected = sum(r["status"] == "rejected" for r in info_rows)
    print(f"Saved features for {len(ids)} clips ({rejected} rejected) -> {FEATURES_NPZ.relative_to(C.PROJECT_ROOT)}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing scripts/extract_features.py


### `scripts/generate_default_labels.py`: generate training labels from categories

Maps each ESC-50 category to plausible musical roles, giving 294 weak labels for training without manual annotation.

In [18]:
%%writefile scripts/generate_default_labels.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Generate role labels from ESC-50 category names.

These are category-level defaults, not per-clip human judgments. Each sound category
is mapped to the roles it would most plausibly fill in a beat. This gives the training
pipeline enough signal to learn from the 1024-d YAMNet embeddings.

    python scripts/generate_default_labels.py
"""
from __future__ import annotations

import csv
import sys
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from encore_audio import config as C
from encore_audio.dataset import LABELS_DIR, LABEL_FIELDS, read_manifest

# Map each ESC-50 category to plausible musical roles (5-role system).
# Roles: percussion, rhythmic, melodic, ambient, transition
CATEGORY_ROLES: dict[str, list[str]] = {
    # Hits and impacts -> percussion
    "door_wood_knock":   ["percussion"],
    "clapping":          ["percussion", "rhythmic"],
    "mouse_click":       ["percussion", "rhythmic"],
    "clock_tick":        ["rhythmic"],
    "glass_breaking":    ["transition", "percussion"],
    "footsteps":         ["percussion", "rhythmic"],
    "fireworks":         ["transition", "percussion"],
    "sneezing":          ["percussion", "transition"],
    "coughing":          ["percussion"],
    "drinking_sipping":  ["rhythmic", "ambient"],
    "toilet_flush":      ["ambient", "transition"],
    "brushing_teeth":    ["rhythmic", "ambient"],
    "can_opening":       ["transition", "percussion"],
    "keyboard_typing":   ["rhythmic"],
    "hand_saw":          ["ambient", "rhythmic"],

    # Sustained / tonal -> melodic or ambient
    "church_bells":      ["melodic", "ambient"],
    "clock_alarm":       ["melodic", "transition"],
    "siren":             ["melodic", "transition"],
    "car_horn":          ["melodic", "transition"],
    "engine":            ["ambient"],
    "vacuum_cleaner":    ["ambient"],
    "washing_machine":   ["ambient", "rhythmic"],
    "train":             ["ambient", "transition"],
    "helicopter":        ["ambient", "rhythmic"],
    "airplane":          ["ambient"],
    "chainsaw":          ["ambient"],

    # Nature / ambience -> ambient
    "rain":              ["ambient"],
    "sea_waves":         ["ambient"],
    "wind":              ["ambient"],
    "thunderstorm":      ["ambient", "transition"],
    "water_drops":       ["rhythmic", "ambient"],
    "crackling_fire":    ["ambient", "rhythmic"],
    "pouring_water":     ["ambient"],

    # Animals
    "dog":               ["percussion", "transition"],
    "cat":               ["melodic", "transition"],
    "rooster":           ["melodic", "transition"],
    "hen":               ["rhythmic", "percussion"],
    "crow":              ["melodic", "transition"],
    "frog":              ["melodic", "rhythmic"],
    "chirping_birds":    ["melodic", "rhythmic"],
    "insects":           ["ambient", "rhythmic"],
    "crickets":          ["rhythmic", "ambient"],
    "sheep":             ["ambient", "melodic"],
    "cow":               ["melodic", "ambient"],
    "pig":               ["percussion", "transition"],

    # Human sounds
    "crying_baby":       ["melodic", "transition"],
    "laughing":          ["percussion", "transition"],
    "breathing":         ["ambient"],
    "snoring":           ["ambient"],
}

def main() -> int:
    manifest = read_manifest()
    if not manifest:
        print("No manifest found. Run scripts/add_recordings.py first.")
        return 1

    LABELS_DIR.mkdir(parents=True, exist_ok=True)
    out_path = LABELS_DIR / "labels_category_defaults.csv"
    now = datetime.now(timezone.utc).isoformat(timespec="seconds")
    written = 0

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=LABEL_FIELDS)
        w.writeheader()
        for row in manifest:
            cat = row["category"]
            roles = CATEGORY_ROLES.get(cat)
            if not roles:
                continue
            # Validate role IDs
            for r in roles:
                assert r in C.ROLE_IDS, f"Unknown role {r!r} for category {cat!r}"
            w.writerow({
                "clip_id": row["clip_id"],
                "rater": "category_defaults",
                "roles": ";".join(roles),
                "best_role": roles[0],
                "unusable": "0",
                "labeled_at": now,
            })
            written += 1

    print(f"Wrote {written} labels to {out_path}")
    print(f"Categories without a mapping: "
          f"{sorted(set(r['category'] for r in manifest) - set(CATEGORY_ROLES))}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing scripts/generate_default_labels.py


### `scripts/evaluate_sound_labels.py`: YAMNet label accuracy

Top-1 and top-5 hit rates for YAMNet's labels on clips whose category we know.

In [19]:
%%writefile scripts/evaluate_sound_labels.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Evaluate YAMNet's sound labels on clips whose true category we know.

    python scripts/evaluate_sound_labels.py

YAMNet is used as-is (no training), so there is nothing to leak: every clip is a test clip.
The only human input is data/label_expectations.csv, which says for each of our categories
which YAMNet class names count as a correct answer. REVIEW THAT FILE: it is a judgement call,
and several categories (bottle, pen, velcro, ...) have no exact YAMNet class at all, so the
best possible answer is only a related class.

Reports top-1 and top-5 hit rates with 95 % Wilson intervals, overall, per source and per
category, and separately for categories that do / do not have an exact YAMNet class.
Writes reports/sound_label_eval.md.
"""
from __future__ import annotations

import csv
import sys
from collections import defaultdict
from math import sqrt
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import numpy as np  # noqa: E402

from encore_audio import config as C  # noqa: E402
from encore_audio.dataset import DATA_DIR, load_features, read_manifest  # noqa: E402
from encore_audio.yamnet_model import class_names  # noqa: E402


def wilson(k: int, n: int) -> str:
    if n == 0:
        return "n/a"
    z, p = 1.96, k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return f"{k}/{n} = {p:.0%} [{max(centre - half, 0):.0%}, {min(centre + half, 1):.0%}]"


def main() -> int:
    names = class_names()
    index = {n: i for i, n in enumerate(names)}
    expectations = {}
    with open(DATA_DIR / "label_expectations.csv", newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            labels = [s.strip() for s in row["acceptable_labels"].split(";") if s.strip()]
            unknown = [s for s in labels if s not in index]
            if unknown:
                print(f"label_expectations.csv, group {row['group']}: not YAMNet class names: {unknown}")
                return 1
            expectations[row["group"]] = ({index[s] for s in labels}, row["exact_class_exists"].strip() == "yes")

    feats = load_features()
    row_of = {cid: i for i, cid in enumerate(feats["clip_ids"].tolist())}
    results = []  # (source, group, exact, top1, top5, top1_name)
    for m in read_manifest():
        if m["group"] not in expectations or m["clip_id"] not in row_of:
            continue
        ok, exact = expectations[m["group"]]
        order = np.argsort(feats["yamnet_scores"][row_of[m["clip_id"]]])[::-1]
        results.append((m["source"], m["group"], exact, int(order[0]) in ok,
                        bool(ok & set(order[:5].tolist())), names[int(order[0])]))
    if not results:
        print("No clips in the manifest belong to a group listed in data/label_expectations.csv.")
        return 1

    def line(title, rows):
        return f"| {title} | {wilson(sum(r[3] for r in rows), len(rows))} | {wilson(sum(r[4] for r in rows), len(rows))} |"

    md = ["# YAMNet sound-label evaluation", "",
          "A clip counts as a hit when one of the acceptable YAMNet classes for its category "
          "(data/label_expectations.csv) is the top-1 label / among the top-5 labels.",
          "Numbers are `hits/clips = rate [95 % Wilson interval]`.", "",
          "| subset | top-1 | top-5 |", "|---|---|---|", line("**all clips**", results)]
    for src in sorted({r[0] for r in results}):
        md.append(line(f"source = {src}", [r for r in results if r[0] == src]))
    md.append(line("categories WITH an exact YAMNet class", [r for r in results if r[2]]))
    if any(not r[2] for r in results):
        md.append(line("categories WITHOUT one (related class only)", [r for r in results if not r[2]]))
    md += ["", "| category | top-1 | top-5 | most common top-1 label |", "|---|---|---|---|"]
    by_group = defaultdict(list)
    for r in results:
        by_group[r[1]].append(r)
    for g in sorted(by_group):
        rows = by_group[g]
        common = max({r[5] for r in rows}, key=lambda n: sum(r[5] == n for r in rows))
        md.append(f"| {g} | {sum(r[3] for r in rows)}/{len(rows)} | {sum(r[4] for r in rows)}/{len(rows)} | {common} |")
    md += ["", f"With {len(results)} clips these intervals are wide. Per-category rows (a handful of clips "
               "each) are anecdotes, not measurements.", ""]
    out = C.PROJECT_ROOT / "reports"
    out.mkdir(exist_ok=True)
    (out / "sound_label_eval.md").write_text("\n".join(md))
    print("\n".join(md))
    print("Saved reports/sound_label_eval.md")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing scripts/evaluate_sound_labels.py


### `scripts/evaluate_roles.py`: role evaluation

Leave-one-category-out evaluation of the role suggestions against human labels, with baselines.

In [20]:
%%writefile scripts/evaluate_roles.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Honest evaluation of musical-role suggestions against human labels.

    python scripts/evaluate_roles.py

Needs: data/manifest.csv, data/labels/labels_*.csv (from label_app.py) and
data/features/features.npz (from extract_features.py).

HOW IT AVOIDS LEAKAGE
Leave-one-group-out cross-validation: for every sound category in turn, the model is trained
on all OTHER categories and tested on that one. So every number below is measured on a sound
category the model has never seen in training, which is the situation Encore is actually in.
(Clips of the same real-world sound from different sources share a group; see dataset.py.)

WHAT IS COMPARED
    random            expected score of a uniformly random ranking (chance level)
    popularity        always suggest the most frequently accepted roles (learned per fold)
    acoustic_prior    the hand-written rules in role_prior.py (no training)
    learned:desc      logistic regression on the 19 acoustic descriptors
    learned:emb       logistic regression on YAMNet's 1024-d embedding
    learned:emb+desc  logistic regression on both

Each metric is reported with a 95 % interval from resampling whole categories. Wide intervals
mean "we do not know yet"; do not quote the middle number without them.

Writes reports/role_eval.md and reports/role_eval.json.
"""
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import numpy as np  # noqa: E402

from encore_audio import config as C  # noqa: E402
from encore_audio.dataset import (aggregate_labels, load_features, rater_agreement,  # noqa: E402
                                  read_labels, read_manifest)
from encore_audio.role_model import FEATURE_SETS  # noqa: E402
from encore_audio.training import (MIN_POSITIVES, REGULARISATION_C, chance_metrics,  # noqa: E402
                                   cluster_bootstrap_ci, fit_role_weights, model_scores, per_clip_metrics)

METRICS = ["hit@2", "precision@2", "recall@2", "top1", "ap"]
REPORTS = C.PROJECT_ROOT / "reports"


def feature_matrix(feats: dict, rows: np.ndarray, feature_set: str) -> np.ndarray:
    emb, desc = feats["embedding_mean"][rows], feats["desc_vector"][rows]
    return {"emb": emb, "desc": desc, "emb+desc": np.hstack([emb, desc])}[feature_set].astype(np.float64)


def load_dataset():
    """Join manifest + aggregated labels + features. Returns dict of aligned arrays."""
    manifest = {r["clip_id"]: r for r in read_manifest()}
    labels = read_labels()
    targets = aggregate_labels(labels)
    feats = load_features()
    row_of = {cid: i for i, cid in enumerate(feats["clip_ids"].tolist())}
    ids = sorted(cid for cid in targets if cid in manifest and cid in row_of)
    if not ids:
        raise SystemExit("No clips have a manifest row, features AND labels yet. "
                         "Label some clips with:  streamlit run label_app.py")
    rows = np.array([row_of[c] for c in ids])
    return {
        "ids": ids, "rows": rows, "feats": feats, "labels": labels,
        "Y": np.array([targets[c]["y"] for c in ids]),
        "groups": np.array([manifest[c]["group"] for c in ids]),
        "sources": np.array([manifest[c]["source"] for c in ids]),
        "prior": feats["prior_scores"][rows].astype(np.float64),
        "n_raters": np.array([targets[c]["n_raters"] for c in ids]),
    }


def cross_validate(data: dict) -> dict:
    """Leave-one-group-out. Returns {method: scores [n, 10]} with out-of-fold scores only."""
    Y, groups, prior = data["Y"], data["groups"], data["prior"]
    n = len(Y)
    n_roles = len(C.ROLE_IDS)
    out = {"popularity": np.zeros((n, n_roles)), "acoustic_prior": prior.copy()}
    out.update({f"learned:{fs}": np.zeros((n, n_roles)) for fs in FEATURE_SETS})
    X = {fs: feature_matrix(data["feats"], data["rows"], fs) for fs in FEATURE_SETS}
    uniq = np.unique(groups)
    for gi, g in enumerate(uniq, 1):
        test, train = groups == g, groups != g
        out["popularity"][test] = Y[train].mean(axis=0)
        for fs in FEATURE_SETS:
            model = fit_role_weights(X[fs][train], Y[train], fs)
            out[f"learned:{fs}"][test] = model_scores(model, X[fs][test], prior[test])
        if gi % 10 == 0 or gi == len(uniq):
            print(f"  fold {gi}/{len(uniq)}")
    return out


def summarise(per_clip: dict, groups: np.ndarray) -> dict:
    res = {}
    for m in METRICS:
        v = per_clip[m]
        if np.all(np.isnan(v)):
            res[m] = None
            continue
        mean, lo, hi = cluster_bootstrap_ci(v, groups)
        res[m] = {"mean": round(mean, 3), "ci95": [round(lo, 3), round(hi, 3)]}
    return res


def fmt(cell) -> str:
    return "n/a" if cell is None else f"{cell['mean']:.2f} [{cell['ci95'][0]:.2f}, {cell['ci95'][1]:.2f}]"


def table(results: dict, title: str) -> list[str]:
    lines = [f"### {title}", "", "| method | " + " | ".join(METRICS) + " |", "|---|" + "---|" * len(METRICS)]
    lines += [f"| {name} | " + " | ".join(fmt(r[m]) for m in METRICS) + " |" for name, r in results.items()]
    return lines + [""]


def main() -> int:
    data = load_dataset()
    Y, groups = data["Y"], data["groups"]
    n, n_groups = len(Y), len(np.unique(groups))
    print(f"{n} labelled clips in {n_groups} groups.")
    if n_groups < 5:
        print("Fewer than 5 groups have labels: leave-one-group-out is not meaningful yet. Label more categories.")
        return 1

    scores = cross_validate(data)
    per_clip = {"random": chance_metrics(Y)}
    per_clip.update({name: per_clip_metrics(s, Y) for name, s in scores.items()})
    overall = {name: summarise(pc, groups) for name, pc in per_clip.items()}

    # Paired comparison of each learned variant against the prior, on the same clips.
    versus_prior = {}
    for fs in FEATURE_SETS:
        diff = per_clip[f"learned:{fs}"]["ap"] - per_clip["acoustic_prior"]["ap"]
        mean, lo, hi = cluster_bootstrap_ci(diff, groups)
        versus_prior[fs] = {"ap_gain": round(mean, 3), "ci95": [round(lo, 3), round(hi, 3)]}

    support = Y.sum(axis=0).astype(int)
    agreement = rater_agreement({c: data["labels"][c] for c in data["ids"]})
    weak_roles = [C.ROLE_NAMES[r] for r, s in zip(C.ROLE_IDS, support) if s < 2 * MIN_POSITIVES]

    md = ["# Musical-role evaluation", "",
          f"Generated {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC, encore {C.VERSION}, "
          f"preprocessing {C.PREPROCESS_VERSION}.", "",
          f"- **{n} labelled clips, {n_groups} sound categories (groups)**, "
          f"{int(np.median(data['n_raters']))} rater(s) per clip (median).",
          f"- Acceptable roles per clip: mean {Y.sum(axis=1).mean():.2f}.",
          "- Protocol: leave-one-category-out. Every score is on a category unseen in training.",
          f"- Learned models: logistic regression, C={REGULARISATION_C}, balanced class weights, "
          "no hyperparameter tuning.",
          "- Numbers are `mean [95 % interval]`, intervals from resampling categories.", ""]
    if agreement.get("n_pairs"):
        md += [f"- Human agreement ({agreement['n_pairs']} rater pairs): two raters share at least one "
               f"role on {agreement['share_at_least_one_role']:.0%} of clips, mean Jaccard overlap "
               f"{agreement['mean_jaccard']:.2f}. No model can be expected to beat this.", ""]
    else:
        md += ["- **Only one rater per clip, so human agreement is unknown.** Have a second person "
               "label at least 50 of the same clips.", ""]
    md += table(overall, "All clips")
    by_source = {}
    for src in np.unique(data["sources"]):
        mask = data["sources"] == src
        if mask.sum() < 10 or len(np.unique(groups[mask])) < 3:
            continue
        by_source[src] = {name: summarise({m: pc[m][mask] for m in METRICS}, groups[mask])
                          for name, pc in per_clip.items()}
        md += table(by_source[src], f"Source = {src} ({int(mask.sum())} clips)")
    md += ["### Learned model versus acoustic prior (paired, average precision)", ""]
    md += [f"- learned:{fs}: {v['ap_gain']:+.3f} [{v['ci95'][0]:+.3f}, {v['ci95'][1]:+.3f}]"
           for fs, v in versus_prior.items()]
    md += ["", "An interval that includes 0 means the data cannot yet tell the two apart.", "",
           "### Label support per role", "", "| role | clips where acceptable |", "|---|---|"]
    md += [f"| {C.ROLE_NAMES[r]} | {s} |" for r, s in zip(C.ROLE_IDS, support)]
    md += [""]
    if weak_roles:
        md += [f"**Too few examples to trust any per-role conclusion for: {', '.join(weak_roles)}.** "
               f"Roles with fewer than {MIN_POSITIVES} examples in a training fold are scored by the acoustic prior.", ""]
    if n < 150 or n_groups < 20:
        md += [f"**Small-data warning:** {n} clips / {n_groups} categories is not enough for stable "
               "estimates. Treat everything above as a rough indication.", ""]
    md += ["### Caveats", "",
           "- hit@2 is easy when raters accept several roles per clip. Always read it next to the "
           "`random` and `popularity` rows.",
           "- Three learned variants are compared on the same data. Picking the best one afterwards "
           "makes its score slightly optimistic.",
           "- Labels are opinions of this team's raters, not ground truth.",
           "- **Category-derived labels are weak labels.** These scores are based on automatic "
           "mapping from ESC-50 category names, not human judgments. Real-world accuracy on "
           "user-recorded sounds will differ.",
           "- Public-dataset clips are not smartphone recordings. The per-source tables show how "
           "the team's own recordings behave.", ""]

    REPORTS.mkdir(exist_ok=True)
    (REPORTS / "role_eval.md").write_text("\n".join(md))
    (REPORTS / "role_eval.json").write_text(json.dumps({
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "n_clips": n, "n_groups": n_groups, "protocol": "leave-one-group-out",
        "overall": overall, "by_source": by_source, "learned_vs_prior_ap": versus_prior,
        "support": dict(zip(C.ROLE_IDS, map(int, support))), "rater_agreement": agreement,
        "label_fingerprint": hash_labels(data),
    }, indent=2))
    print("\n".join(md))
    print("Saved reports/role_eval.md and reports/role_eval.json")
    return 0


def hash_labels(data: dict) -> str:
    """Fingerprint of the exact labelled set, so train_roles.py can tell whether the saved
    evaluation belongs to the data it is training on."""
    import hashlib

    h = hashlib.sha256()
    for cid, y in zip(data["ids"], data["Y"]):
        h.update(cid.encode() + bytes(y.tolist()))
    return h.hexdigest()[:16]


if __name__ == "__main__":
    raise SystemExit(main())

Writing scripts/evaluate_roles.py


### `scripts/train_roles.py`: train and install the role model

Trains on all labelled clips and installs the model, unless the evaluation says it is worse than the rules.

In [21]:
%%writefile scripts/train_roles.py
# Written by Encore_YAMNet.ipynb. Edit the code in the notebook, not in this file.
"""Train the final role model on ALL labelled clips and install it for inference.

    python scripts/evaluate_roles.py      # first: look at the evidence
    python scripts/train_roles.py         # then: train and install

Writes models/role_model.npz + models/role_model.json. classify_audio_file() picks the model
up automatically the next time it runs. To go back to the acoustic prior, delete those two
files (or set the environment variable ENCORE_ROLE_METHOD=prior).

The model trained here is the same recipe that evaluate_roles.py cross-validated; the
evaluation summary is copied into role_model.json so the numbers travel with the weights.
"""
from __future__ import annotations

import argparse
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import numpy as np  # noqa: E402

from encore_audio import config as C  # noqa: E402
from encore_audio.role_model import FEATURE_SETS, save_role_model  # noqa: E402
from encore_audio.training import MIN_POSITIVES, REGULARISATION_C, fit_role_weights  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parent))
from evaluate_roles import feature_matrix, hash_labels, load_dataset  # noqa: E402

MIN_CLIPS, MIN_GROUPS = 60, 10


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--features", choices=FEATURE_SETS, default="emb+desc")
    ap.add_argument("--force", action="store_true", help="train even with very little data")
    args = ap.parse_args()

    data = load_dataset()
    Y, groups = data["Y"], data["groups"]
    n, n_groups = len(Y), len(np.unique(groups))
    print(f"{n} labelled clips in {n_groups} groups.")
    if (n < MIN_CLIPS or n_groups < MIN_GROUPS) and not args.force:
        print(f"That is below the minimum of {MIN_CLIPS} clips / {MIN_GROUPS} groups for a model worth "
              "installing. Keep using the acoustic prior and label more clips (or pass --force).")
        return 1

    model = fit_role_weights(feature_matrix(data["feats"], data["rows"], args.features), Y, args.features)
    if model is None:
        print(f"No role has at least {MIN_POSITIVES} positive and {MIN_POSITIVES} negative examples. Nothing to train.")
        return 1

    evaluation = None
    report = C.PROJECT_ROOT / "reports" / "role_eval.json"
    if report.is_file():
        rep = json.loads(report.read_text())
        if rep.get("label_fingerprint") == hash_labels(data):
            evaluation = {"protocol": rep["protocol"], "n_clips": rep["n_clips"], "n_groups": rep["n_groups"],
                          "this_model": rep["overall"].get(f"learned:{args.features}"),
                          "acoustic_prior": rep["overall"].get("acoustic_prior"),
                          "random": rep["overall"].get("random"),
                          "ap_gain_vs_prior": rep["learned_vs_prior_ap"].get(args.features)}
            gain = evaluation["ap_gain_vs_prior"]
            if gain and gain["ci95"][1] < 0 and not args.force:
                print(f"On unseen categories this model is clearly WORSE than the acoustic prior "
                      f"(AP gain {gain['ap_gain']:+.3f}, 95 % interval {gain['ci95']}).\n"
                      "Not installing it. Keep the prior, or pass --force if you really want it.")
                return 1
            if gain and gain["ci95"][0] <= 0:
                print(f"WARNING: on unseen categories this model is not clearly better than the acoustic prior "
                      f"(AP gain {gain['ap_gain']:+.3f}, 95 % interval {gain['ci95']}). Installing it anyway; "
                      "delete models/role_model.* to go back to the prior.")
        else:
            print("NOTE: reports/role_eval.json was made from different labels. Re-run evaluate_roles.py; "
                  "no evaluation summary is attached to this model.")
    else:
        print("NOTE: no evaluation report found. Run scripts/evaluate_roles.py to know how good this model is.")

    uncovered = [r for r in C.ROLE_IDS if r not in model.role_ids]
    meta = dict(model.meta, trained_at=datetime.now(timezone.utc).isoformat(timespec="seconds"),
                n_clips=n, n_groups=n_groups, n_raters=int(np.median(data["n_raters"])),
                model="one-vs-rest logistic regression on standardised features",
                regularisation_C=REGULARISATION_C, class_weight="balanced",
                support=dict(zip(C.ROLE_IDS, map(int, Y.sum(axis=0)))),
                roles_left_to_acoustic_prior=uncovered, evaluation=evaluation)
    save_role_model(model.mean, model.scale, model.coef, model.intercept, meta)
    print(f"Installed models/role_model.npz + role_model.json  (features: {args.features})")
    print(f"Learned roles: {', '.join(model.role_ids)}")
    if uncovered:
        print(f"Too few labelled examples, left to the acoustic prior: {', '.join(uncovered)}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing scripts/train_roles.py


## A4. Check that everything written above works

Takes about a minute. It should end with `All checks passed.`

In [22]:
from encore_audio.yamnet_model import download_yamnet

download_yamnet()                                # about 17 MB, first run only
!{sys.executable} scripts/self_check.py

YAMNet saved to /content/models/yamnet
1. YAMNet itself (same checks as Google's own yamnet_test.py)
I0000 00:00:1794710527.406494   10890 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794710534.776556   10890 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1794710540.442626   10890 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
  [PASS] sine.wav -> 'Sine wave' in top 10  (Telephone, Busy signal, Dial tone)
  [PASS] white.wav -> 'White noise' in top 10  (Water, Waterfall, White noise)
2. Output contract
  [PASS] result is JSON-serialisable
  [PASS] 5 sound labels
  [PASS] 2 role suggestions
  [PASS] all 5 roles listed in canonical order
  [PASS] role scores are within [0, 1]
3. Formats
  [PASS] .m4a decodes and classifies
  [PASS] .mp3 decodes and classifies
4. Bad input is rejected cleanly
  [PASS] silent.wav -> rejected a

## A5. Get the ESC-50 data

The 300 prepared ESC-50 clips are already in the repo.
On Colab, the cell below downloads the ESC-50 dataset, samples 300 clips and creates the
manifest and label expectations file. **Skip this cell if you already have the data.**

In [23]:
import csv as _csv
from pathlib import Path as _P

# Write label_expectations.csv (what YAMNet labels count as correct for our categories)
if not _P("data/label_expectations.csv").is_file():
    _P("data/label_expectations.csv").write_text('''\
group,acceptable_labels,exact_class_exists,note
bottle,"Glass;Chink, clink;Tap",no,YAMNet has no bottle class; these are the closest related classes
clap,"Clapping;Hands;Slap, smack",yes,
coins,"Coin (dropping);Jingle, tinkle;Chink, clink",yes,
keys,"Keys jangling;Jingle, tinkle;Chink, clink",yes,
knock,"Knock;Door;Tap;Thump, thud",yes,
paper,"Crumpling, crinkling;Tearing;Rustle",no,No paper class; depends on what was done with the paper
pen,"Clicking;Tick;Writing;Tap;Scratch",no,No pen class; depends on click vs tap vs writing
snap,"Finger snapping;Clicking",yes,
water,"Water;Pour;Liquid;Trickle, dribble;Drip;Fill (with liquid);Splash, splatter;Water tap, faucet",yes,
zipper,"Zipper (clothing)",yes,
keyboard_typing,"Typing;Computer keyboard;Typewriter",yes,
soda_can_opening,"Hiss;Burst, pop",no,No can-opening class
tongue_clicking,"Clicking;Tick",no,No tongue-click class
velcro,"Tearing",no,No velcro class
whistling,"Whistling;Whistle",yes,
''')
    print("Wrote data/label_expectations.csv")
else:
    print("data/label_expectations.csv already exists, skipping.")

# Download and prepare ESC-50 clips (Colab only, about 600 MB download)
if ON_COLAB and not list(_P("data/clips/esc50").glob("*.wav")):
    print("Downloading ESC-50 dataset (about 600 MB)...")
    !git clone --depth 1 https://github.com/karolpiczak/ESC-50.git data/raw/ESC-50
    print("Preparing 300 clips (6 per class)...")
    !{sys.executable} scripts/prepare_esc50.py
elif list(_P("data/clips/esc50").glob("*.wav")):
    print(f"ESC-50 clips already present ({len(list(_P('data/clips/esc50').glob('*.wav')))} files).")
    # Make sure manifest exists
    if not _P("data/manifest.csv").is_file() or sum(1 for _ in open("data/manifest.csv")) < 10:
        !{sys.executable} scripts/prepare_esc50.py
else:
    print("Not on Colab and no ESC-50 clips found. Run from the repo or clone ESC-50 manually.")

Wrote data/label_expectations.csv
Cloning into 'data/raw/ESC-50'...
remote: Enumerating objects: 2018, done.
remote: Counting objects: 100% (2018/2018), done.
remote: Compressing objects: 100% (1798/1798), done.
remote: Total 2018 (delta 215), reused 2009 (delta 214), pack-reused 0 (from 0)
Receiving objects: 100% (2018/2018), 615.62 MiB | 22.05 MiB/s, done.
Resolving deltas: 100% (215/215), done.
Updating files: 100% (2011/2011), done.
Preparing 300 clips (6 per class)...
Prepared 300 ESC-50 clips from 50 classes (0 silent/unreadable clips skipped).
Manifest: 300 new rows, 300 total -> data/manifest.csv


## A6. Generate training labels

The cell below maps each ESC-50 category name to its plausible musical roles (e.g. "clapping"
maps to percussion + rhythmic). This gives 294 weak labels, enough to train the learned model
without manual annotation. These are **category-level defaults, not per-clip human judgments**,
so treat the resulting accuracy numbers as an upper bound on real-world performance.

In [24]:
!{sys.executable} scripts/generate_default_labels.py

Wrote 294 labels to /content/data/labels/labels_category_defaults.csv
Categories without a mapping: ['door_wood_creaks']


# Part B: using it

## B1. Classify one sound

`classify_audio_file(path)` is the one function the backend calls. It returns a plain dict that is
ready for `json.dumps`. Change `SAMPLE` to any audio file (m4a, wav, mp3) to try your own.

In [25]:
from encore_audio import classify_audio_file, warm_up, ROLES
from encore_audio.yamnet_model import class_names

warm_up()
import glob as _glob
_clips = sorted(_glob.glob("data/clips/esc50/esc50__door_wood_knock__*.wav"))
if not _clips:
    _clips = sorted(_glob.glob("data/clips/esc50/*.wav"))
SAMPLE = _clips[0]
print("Demo clip:", SAMPLE)

display(Audio(SAMPLE))
result = classify_audio_file(SAMPLE)

print("status:", result["status"], "| sound character:", ", ".join(result["sound_character"]))
print("review flags:", result["review"]["flags"] or "none")
print("role method:", result["method"]["roles"])
display(pd.DataFrame(result["sound_labels"])[["rank", "label", "score"]].style.hide(axis="index").set_caption("Top 5 YAMNet labels"))
display(pd.DataFrame(result["role_suggestions"])[["rank", "role", "score", "strength", "source"]].style.hide(axis="index").set_caption("Top 2 role suggestions"))
display(pd.DataFrame(result["all_roles"])[["role", "score", "strength", "suggested"]].style.hide(axis="index").set_caption("All five roles (the user can pick any)"))

Demo clip: data/clips/esc50/esc50__door_wood_knock__1-101336-A-30.wav


status: ok | sound character: series of hits, short, low / dark, unpitched / noisy
review flags: none
role method: acoustic-prior-v0


rank,label,score
1,Knock,0.520100
2,Door,0.295900
3,Drum machine,0.154100
4,Percussion,0.145700
5,Tap,0.138300


rank,role,score,strength,source
1,Percussion,1.000000,strong,acoustic_prior
2,Rhythmic,0.464600,moderate,acoustic_prior


role,score,strength,suggested
Percussion,1.000000,strong,True
Rhythmic,0.464600,moderate,True
Melodic,0.125000,weak,False
Ambient,0.123600,weak,False
Transition,0.125000,weak,False


## B2. What happens inside, step by step

The same pipeline as above, one stage at a time, so every number can be explained.

**Load and clean the audio.** ffmpeg decodes any format to 16 kHz mono (what YAMNet requires).
Silent, corrupt or tiny files are rejected here. Leading and trailing silence is trimmed and the
level is normalised.

In [26]:
from encore_audio.audio_io import load_audio

prepared = load_audio(SAMPLE)
print("samples kept for analysis:", prepared.active.size, f"({prepared.active.size / 16000:.2f} s at 16 kHz)")
print("flags raised by preprocessing:", prepared.flags or "none")
pd.Series(prepared.info, name="value").to_frame()

samples kept for analysis: 46240 (2.89 s at 16 kHz)
flags raised by preprocessing: none


,value
file_duration_s,2.89
analysed_duration_s,2.89
sample_rate,16000.00
peak_dbfs,-0.90
noise_floor_dbfs,-54.20
contrast_db,46.40
clipped_fraction,0.00
gain_applied_db,-0.00


**Run YAMNet.** YAMNet slides a 0.96 s window over the audio every 0.48 s. For each window
("frame") it gives 521 class scores and a 1,024-number embedding. We average over frames: the
averaged scores give the sound labels, and the averaged embedding is the input to the learned
role model.

In [27]:
from encore_audio.yamnet_model import run_yamnet

scores, embeddings = run_yamnet(prepared.yamnet_input)
print("scores:", scores.shape, "= (frames, classes)   embeddings:", embeddings.shape, "= (frames, dimensions)")

names = class_names()
mean_scores = scores.mean(axis=0)
top = np.argsort(mean_scores)[::-1][:5]
per_frame = pd.DataFrame(scores[:, top].round(2), columns=[names[k] for k in top])
per_frame.index.name = "frame"
per_frame.loc["mean"] = mean_scores[top].round(2)
per_frame

scores: (5, 521) = (frames, classes)   embeddings: (5, 1024) = (frames, dimensions)


,Knock,Door,Drum machine,Percussion,Tap
frame,,,,,
0,0.64,0.52,0.13,0.08,0.23
1,0.03,0.18,0.01,0.06,0.12
2,0.64,0.14,0.11,0.28,0.07
3,0.81,0.41,0.20,0.10,0.19
4,0.48,0.22,0.33,0.20,0.09
mean,0.52,0.30,0.15,0.15,0.14


**Measure the sound itself.** These numbers describe the sound, not the object, which is why
they work for objects nobody has recorded before.

In [28]:
from encore_audio.descriptors import compute_descriptors

descriptors = compute_descriptors(prepared.active)
pd.Series(descriptors, name="value").round(3).to_frame()

,value
duration_s,2.890
effective_duration_s,0.787
event_duration_s,0.049
attack_ms,2.000
decay_ms,45.000
sustain_ratio,0.102
temporal_centroid,0.427
onset_count,11.000
onset_rate_hz,3.806
centroid_hz,262.454


**Score the five roles.** Until we have human role labels, scores come from the acoustic prior:
measurements become traits between 0 and 1, and traits become role scores (percussion = a hit AND
short). This is hand-written logic, not machine learning. B5 is where it gets measured.

In [29]:
from encore_audio.role_prior import compute_traits, prior_scores

traits = compute_traits(descriptors)
display(pd.Series(traits, name="trait (0 to 1)").round(2).to_frame().T)

role_scores = prior_scores(descriptors)
pd.Series(role_scores, name="acoustic prior score").round(3).sort_values(ascending=False).to_frame()

,hit,short,very_short,long,pitched,unpitched,dark,bright,busy,moving,steady,low_pitch,high_pitch,long_tail
trait (0 to 1),1.0,1.0,1.0,0.48,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


,acoustic prior score
percussion,1.000
rhythmic,0.465
melodic,0.125
transition,0.125
ambient,0.124


## B3. Run every clip in the dataset

`data/manifest.csv` lists every clip we have: 300 ESC-50 clips (50 everyday sound categories) plus
our own recordings once they are registered.

To add our own recordings, put them in `data/my_recordings/<category>/` (one folder per sound,
any of m4a / wav / mp3) and run the next cell. It is safe to run again.

In [30]:
!{sys.executable} scripts/add_recordings.py

No audio files found in sub-folders of /content/data/my_recordings.


In [31]:
# Runs each clip through the exact pipeline from B2 and caches the results (about 0.1 s per clip).
!{sys.executable} scripts/extract_features.py

info = pd.read_csv("data/features/clip_info.csv")
print(len(info), "clips |", (info["status"] == "rejected").sum(), "rejected")
info[["clip_id", "source", "category", "yamnet_top5", "prior_top3", "note"]].sample(8, random_state=320)

I0000 00:00:1794710645.226216   13979 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794710648.614964   13979 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1794710652.461312   13979 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
  50/300 clips
  100/300 clips
  150/300 clips
  200/300 clips
  250/300 clips
  300/300 clips
Saved features for 300 clips (0 rejected) -> data/features/features.npz
300 clips | 0 rejected


,clip_id,source,category,yamnet_top5,prior_top3,note
289,esc50__water_drops__2-139748-A-15,esc50,water_drops,Music (0.42) | Wood block (0.26) | Percussion ...,percussion | ambient | melodic,NaN
126,esc50__drinking_sipping__1-37226-A-29,esc50,drinking_sipping,Animal (0.14) | Pig (0.13) | Oink (0.11) | Ins...,percussion | transition | melodic,NaN
57,esc50__clapping__3-149465-A-22,esc50,clapping,Applause (0.17) | Clapping (0.14) | Speech (0....,ambient | rhythmic | percussion,steady_or_noisy
100,esc50__crow__5-214869-A-9,esc50,crow,Crow (0.90) | Caw (0.81) | Animal (0.75) | Wil...,ambient | melodic | transition,steady_or_noisy
83,esc50__cow__5-194899-D-3,esc50,cow,"Cattle, bovinae (0.55) | Moo (0.50) | Livestoc...",melodic | ambient | transition,NaN
19,esc50__can_opening__1-60676-A-34,esc50,can_opening,Silence (0.34) | Cap gun (0.15) | Camera (0.11...,percussion | rhythmic | transition,NaN
149,esc50__footsteps__5-243025-A-25,esc50,footsteps,Crack (0.55) | Crackle (0.38) | Breaking (0.30...,percussion | rhythmic | transition,NaN
14,esc50__brushing_teeth__2-83536-A-27,esc50,brushing_teeth,Breathing (0.20) | Pant (0.12) | Writing (0.10...,percussion | rhythmic | transition,NaN


## B4. How good are YAMNet's sound labels?

YAMNet is not trained here, so every clip is a fair test clip. `data/label_expectations.csv` says
which YAMNet class names count as correct for each of our categories. That file is a judgement
call, so read it before quoting these numbers. Several of our categories (bottle, pen, velcro,
soda can, tongue click, paper) have no YAMNet class at all.

In [32]:
display(pd.read_csv("data/label_expectations.csv"))
!{sys.executable} scripts/evaluate_sound_labels.py

,group,acceptable_labels,exact_class_exists,note
0,bottle,"Glass;Chink, clink;Tap",no,YAMNet has no bottle class; these are the clos...
1,clap,"Clapping;Hands;Slap, smack",yes,NaN
2,coins,"Coin (dropping);Jingle, tinkle;Chink, clink",yes,NaN
3,keys,"Keys jangling;Jingle, tinkle;Chink, clink",yes,NaN
4,knock,"Knock;Door;Tap;Thump, thud",yes,NaN
5,paper,"Crumpling, crinkling;Tearing;Rustle",no,No paper class; depends on what was done with ...
6,pen,Clicking;Tick;Writing;Tap;Scratch,no,No pen class; depends on click vs tap vs writing
7,snap,Finger snapping;Clicking,yes,NaN
8,water,"Water;Pour;Liquid;Trickle, dribble;Drip;Fill (...",yes,NaN
9,zipper,Zipper (clothing),yes,NaN


I0000 00:00:1794710704.128636   16625 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794710707.656248   16625 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1794710709.390255   16625 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
# YAMNet sound-label evaluation

A clip counts as a hit when one of the acceptable YAMNet classes for its category (data/label_expectations.csv) is the top-1 label / among the top-5 labels.
Numbers are `hits/clips = rate [95 % Wilson interval]`.

| subset | top-1 | top-5 |
|---|---|---|
| **all clips** | 17/30 = 57% [39%, 73%] | 21/30 = 70% [52%, 83%] |
| source = esc50 | 17/30 = 57% [39%, 73%] | 21/30 = 70% [52%, 83%] |
| categories WITH an exact YAMNet class | 17/24 = 71% [51%, 85%] | 21/24 = 88% [69%, 96%] |
| categories WITHOUT one (related class only) | 0/6 = 0% [0%, 39%] | 0/6 = 0% [0%, 39%] 

In [33]:
from encore_audio.dataset import read_labels, read_manifest

labels = read_labels()
raters = sorted({r for by_rater in labels.values() for r in by_rater})
print(f"{len(read_manifest())} clips in the manifest, {len(labels)} have at least one label, "
      f"{sum(len(v) >= 2 for v in labels.values())} have two or more raters.")
for r in raters:
    print(f"  {r}: {sum(r in by_rater for by_rater in labels.values())} clips")
if not labels:
    print("No labels yet. The two cells below will say so and stop, which is expected.")

300 clips in the manifest, 294 have at least one label, 0 have two or more raters.
  category_defaults: 294 clips


**Evaluation.** Leave-one-category-out: for each sound category, train on all the others and test
on that one, so every score is on a kind of sound the model never saw. The table compares random
guessing, "always suggest the most popular roles", the acoustic prior and three learned variants,
each with a 95 % interval. Read hit@2 next to the `random` row: with five roles, random guessing
already scores about 0.6.

In [34]:
!{sys.executable} scripts/evaluate_roles.py

294 labelled clips in 49 groups.
  fold 10/49
  fold 20/49
  fold 30/49
  fold 40/49
  fold 49/49
# Musical-role evaluation

Generated 2026-11-15 02:45 UTC, encore 0.1.0, preprocessing pre-1.

- **294 labelled clips, 49 sound categories (groups)**, 1 rater(s) per clip (median).
- Acceptable roles per clip: mean 1.71.
- Protocol: leave-one-category-out. Every score is on a category unseen in training.
- Learned models: logistic regression, C=0.1, balanced class weights, no hyperparameter tuning.
- Numbers are `mean [95 % interval]`, intervals from resampling categories.

- **Only one rater per clip, so human agreement is unknown.** Have a second person label at least 50 of the same clips.

### All clips

| method | hit@2 | precision@2 | recall@2 | top1 | ap |
|---|---|---|---|---|---|
| random | 0.61 [0.58, 0.65] | 0.34 [0.32, 0.37] | 0.40 [0.40, 0.40] | 0.34 [0.32, 0.37] | n/a |
| popularity | 0.51 [0.37, 0.65] | 0.26 [0.18, 0.33] | 0.36 [0.24, 0.47] | 0.51 [0.37, 0.65] | 0.64 [0.56, 0

## B6. Train and install the learned role model

Writes `models/role_model.npz` and `models/role_model.json` (about 100 KB, commit them). It
refuses to install a model that the evaluation showed to be clearly worse than the acoustic
prior. `classify_audio_file` picks the model up automatically.

In [35]:
!{sys.executable} scripts/train_roles.py

294 labelled clips in 49 groups.
Installed models/role_model.npz + role_model.json  (features: emb+desc)
Learned roles: percussion, rhythmic, melodic, ambient, transition


## B7. Check which role method is live

In [36]:
result = classify_audio_file(SAMPLE)
print("role method:", result["method"]["roles"])
print("note:", result["method"].get("role_model_note", "learned model active"))
pd.DataFrame(result["role_suggestions"])[["rank", "role", "score", "strength", "source"]].style.hide(axis="index")

role method: learned-logreg-v1
note: learned model active


rank,role,score,strength,source
1,Percussion,0.919900,strong,learned
2,Rhythmic,0.413400,moderate,learned
